# taskb_encoder_dev

Historical source of the reused conventional seed42 checkpoint. See [Task B guide](README_taskb.md) for report evidence, execution order and historical-source limitations.


In [ ]:
# Đổi seed/run_id có chủ ý trước run mới; không đổi để resume.
CFG = {'preparation_root': '/content/drive/MyDrive/ĐATN/data/processed/benchmark_preparation/FashionIQ/fashioniq_prep_v1_20261103', 'output_root': '/content/drive/MyDrive/ĐATN/data/processed/tme_development/FashionIQ', 'run_id': 'tme_dev_v1_seed42', 'resume': True, 'seed': 42, 'amp': False, 'require_cuda': True, 'active_layers': None, 'overfit_size': 32, 'pilot_train_size': 16204, 'pilot_val_max': 1796, 'batch_size': 4, 'eval_batch_size': 16, 'encode_batch_size': 8, 'cache_shard_size': 16, 'overfit_steps': 80, 'pilot_epochs': 3, 'learning_rate': 0.0001, 'weight_decay': 0.01, 'gradient_clip': 1.0, 'retrieval_temperature': 0.07, 'alignment_temperature': 0.07, 'alignment_weight': 1.0, 'overfit_min_relative_drop': 0.01, 'checkpoint_every_steps': 1024, 'log_every_steps': 20, 'reuse_verified_pooled': True, 'pooled_reuse_atol': 1e-05, 'dataset_root': '/content/fashioniq_data', 'allow_kaggle_download': True, 'image_search_roots': ['/content/fashioniq_data', '/content/drive/MyDrive/ĐATN/data/processed/images'], 'local_cache': '/content/visfit_tme_development_cache', 'max_epochs': 3, 'patience': 2, 'variants': ['conventional', 'no_alignment', 'full'], 'pilot_root': '/content/drive/MyDrive/ĐATN/data/processed/tme_pilot/FashionIQ/tme_pilot_v1_20261106_seed42_iofix01'}

## 1. Bootstrap trước mọi import model
Nếu cần restart, lưu traceback/notebook trước; không chạy tiếp import từ kernel cũ.

In [ ]:
# Chỉ stdlib trước import model; bootstrap fix: dependency-owner-scope-v1.
import sys, os, subprocess, importlib.util, importlib.metadata, json
from pathlib import Path
BOOTSTRAP = {'python':sys.version,'installs':[],'pins':{},'passed':False,'bootstrap_fix':'dependency-owner-scope-v1'}
PINS = {'transformers':'4.57.1','tokenizers':'0.22.1','huggingface-hub':'0.36.0'}
for package in ('torch','numpy','pandas','Pillow'):
    try: BOOTSTRAP['pins'][package]=importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        raise RuntimeError('Colab base package missing: '+package+'; use a standard GPU runtime') from None
needed=[]
for package,version in PINS.items():
    try: installed=importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError: installed=None
    if installed!=version:needed.append(package+'=='+version)
    module={'huggingface-hub':'huggingface_hub'}.get(package,package)
    if installed!=version and any(n==module or n.startswith(module+'.') for n in sys.modules):
        raise RuntimeError('A dependency requiring a version change is already loaded: '+package+'. Save the notebook and use Runtime > Restart session, then run from the top. No automatic install/restart loop.')
    BOOTSTRAP['pins'][package]=version
if importlib.util.find_spec('kaggle') is None:needed.append('kaggle>=1.6.17,<2')
if needed:
    p=subprocess.run([sys.executable,'-m','pip','install','--quiet',*needed],capture_output=True,text=True)
    if p.returncode:raise RuntimeError('Dependency installation failed; stopped before importing models. No torch/CUDA upgrade requested.')
    BOOTSTRAP['installs']=needed
check=subprocess.run([sys.executable,'-m','pip','check'],capture_output=True,text=True)
BOOTSTRAP['pip_check']={'returncode':check.returncode,'report':check.stdout+check.stderr}
# Scope by dependent package owner, not a substring of the dependency mentioned in the error.
# Gradio/Diffusers are not used by this pilot; their conflicts remain visible in the report.
from packaging.requirements import Requirement
from packaging.utils import canonicalize_name
pending=['torch','torchvision','transformers','tokenizers','huggingface-hub','numpy','pandas','Pillow','safetensors','packaging']
core=set()
while pending:
    owner=canonicalize_name(pending.pop())
    if owner in core:continue
    core.add(owner)
    for raw in importlib.metadata.requires(owner) or []:
        req=Requirement(raw)
        if req.marker is None or req.marker.evaluate({'extra':''}):pending.append(req.name)
lines=[line for line in (check.stdout+check.stderr).splitlines() if line.strip() and 'No broken requirements found' not in line]
critical=[line for line in lines if canonicalize_name(line.split()[0]) in core]
BOOTSTRAP['dependency_scope']={'core_packages':sorted(core),'critical_conflicts':critical,'outside_pilot_conflicts':[line for line in lines if line not in critical]}
local=Path('/content/visfit_tme_bootstrap');local.mkdir(parents=True,exist_ok=True)
(local/'bootstrap_report.json').write_text(json.dumps(BOOTSTRAP,indent=2))
if check.returncode and critical:raise RuntimeError('pip check found TME dependency conflicts: '+'; '.join(critical))
probe=subprocess.run([sys.executable,'-c',
    'import torch,transformers; from transformers import CLIPModel,CLIPTokenizer,CLIPImageProcessor; from transformers.generation import GenerationMixin; print(transformers.__version__); print(transformers.__file__)'],capture_output=True,text=True)
BOOTSTRAP['import_probe']={'returncode':probe.returncode,'stdout':probe.stdout,'traceback':probe.stderr}
(local/'bootstrap_report.json').write_text(json.dumps(BOOTSTRAP,indent=2))
if probe.returncode:raise RuntimeError('CLIP dependency import failed in a clean subprocess. Full traceback: /content/visfit_tme_bootstrap/bootstrap_report.json. Diagnose before dataset/training.')
BOOTSTRAP['passed']=True
(local/'bootstrap_report.json').write_text(json.dumps(BOOTSTRAP,indent=2))
print('Bootstrap đạt; giữ nguyên torch/CUDA:',BOOTSTRAP['pins']['torch'])
print('TME dependency conflicts:',len(critical),'| clean CLIP import:',probe.returncode==0,'| packages installed:',BOOTSTRAP['installs'])
if check.returncode:print('pip check có xung đột ngoài dependency graph của pilot; giữ nguyên đầy đủ trong bootstrap_report.json. Không coi toàn môi trường là pip-clean.')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
PREPARATION_ANCHORS = {'data_audit.json': 'fbaa7eae7b99c5e5aeeddcb0bc6027d3870e8c4f9be5658559caa95662321e95', 'feature_metadata.json': '48ee1866dc22f07fa588ea815318ddc9995cfa8734ffd0f2acc88c430f855342', 'extraction_contract.json': 'bbd53b94172aa5c07a358a0697163c620a823b297c6fec6209167bdb59d1f0a1', 'training_plan.json': 'e5496feae33ab0e9dc832e29a1a9097b2d7fed6c2683bd9d4314d6e4973bdc21', 'development_gallery.csv': '83c08f14b606227c1a406b5074163ef70e42eb025144d19e2b1de4f38d234dc5', 'development_val.csv': 'ace52e94935adacfd19aeaea7a99314602954bce9bff43458ceb564fdf05a8b5', 'protocol_config.json': '2fa2322dee2ebe9964e4ce63ebf0cd8dabad83392a39e288d0e43334e40faffb', 'development_train.csv': '465f758604249e2005cc58c0dfa79edf236081613e908c1e5942fd61abf17086', 'feature_views/development_gallery.csv': '283199425c5fc88fb8e063ea28be629e611e44513689e97f22ee3d0ec5a33927', 'feature_views/development_val.csv': 'cbd8d7b72fa7fb4e3c38d64f3bb8ceeeba73be946e9918c5a77c856a42217619', 'feature_views/development_train.csv': '6f7a5e6ab8792dd8fd08d6fea8c472a9d9b56b59af38ba146467d797f8a98ff0'}
SOURCE_REFERENCES = [{'kind': 'paper', 'url': 'https://zhoujiahuan1991.github.io/pub/AAAI2024_FASHIONERN.pdf', 'section': 'Triple-branch Modifier Enhancement, equations 1–5; Network Training', 'sha256': '87a9b3d90bc306f976f928f793fd25a790cb86abe5fc2ba71842c6ea6c053c23'}, {'kind': 'official_repository', 'url': 'https://github.com/ChenAnno/FashionERN_AAAI2024', 'note': 'README explicitly says original TME is not released; vanilla CLIP fallback is not full TME'}, {'kind': 'transformers_source', 'url': 'https://github.com/huggingface/transformers/blob/v4.57.1/src/transformers/models/clip/modeling_clip.py', 'note': 'Native CLIP block, causal/padding masks, vision hidden_states and pooling'}, {'kind': 'locked_caption_policy', 'url': 'https://github.com/ABaldrati/CLIP4Cir/blob/56bb547909d57bd8b5d2ef7ba55149962f77d482/src/validate.py'}]
CODE_SHA256 = 'e6d9e3e0bb215fa5dffad8d58b4e4857826e97fb19d67860c45282111c5dd4fd'


In [ ]:
import os, sys, json, hashlib, math, random, time, copy, traceback, uuid, shutil
import platform, tempfile, contextlib, importlib.metadata, subprocess, zipfile, stat
from pathlib import Path, PurePosixPath
from collections import OrderedDict
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F
from PIL import Image
from transformers import CLIPModel, CLIPTokenizer, CLIPImageProcessor

CATEGORIES = ('dress', 'shirt', 'toptee')
CAPTION_POLICY = 'clip4cir-strip-dot-question-comma-space-capitalize-first-and-v1'
IMPLEMENTATION = 'visfit-tme-development-v1.0.0'

def require(condition, message):
    if not condition: raise RuntimeError(message)

def canonical(x):
    return json.dumps(x, ensure_ascii=False, sort_keys=True, separators=(',', ':'), allow_nan=False).encode()

def fingerprint(x): return hashlib.sha256(canonical(x)).hexdigest()

def file_hash(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(8*1024*1024), b''): h.update(block)
    return h.hexdigest()

def read_json(path): return json.loads(Path(path).read_text())

def write_bytes(path, raw, immutable=False):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    if immutable and path.exists():
        require(path.read_bytes() == raw, 'Existing immutable artifact differs: '+str(path)); return
    tmp = path.with_name(path.name+'.'+uuid.uuid4().hex+'.partial')
    tmp.write_bytes(raw); os.replace(tmp, path)

def write_json(path, value, immutable=False):
    write_bytes(path, canonical(value)+b'\n', immutable)

def write_csv(path, frame, immutable=False):
    write_bytes(path, frame.to_csv(index=False, lineterminator='\n').encode(), immutable)

def safe_child(root, relative):
    root = Path(root).resolve(); path = (root/relative).resolve()
    require(path != root and root in path.parents, 'Path escapes root'); return path

def tensor_hash(module):
    h = hashlib.sha256()
    for name, value in sorted(module.state_dict().items()):
        h.update(name.encode()); h.update(str(value.dtype).encode())
        h.update(value.detach().cpu().contiguous().numpy().tobytes())
    return h.hexdigest()

def finite(x, name):
    require(bool(torch.isfinite(x).all()), 'NaN/Inf: '+name); return x

def unit(x):
    finite(x, 'unnormalized vector')
    require(bool((x.float().norm(dim=-1)>1e-8).all()), 'Zero retrieval vector')
    return F.normalize(x.float(), dim=-1)

def normalize_caption(x): return x.strip('.?, ')

def caption_text(a, b): return normalize_caption(a).capitalize()+' and '+normalize_caption(b)

def assess_captions(caps):
    if not isinstance(caps, list) or len(caps)!=2 or not all(isinstance(x,str) for x in caps):
        return 'schema_error', None
    content = [bool(normalize_caption(x)) and not normalize_caption(x).isspace() for x in caps]
    if not any(content): return 'both_empty_captions', None
    return None, 'single_empty_caption' if not all(content) else None

def stratified_sample(frame, n, seed):
    require(type(n) is int and 3<=n<=len(frame), 'Sample budget outside available range; change CFG explicitly')
    require(frame.query_id.is_unique, 'Duplicate query IDs')
    # Sort by query ID before RNG: stable if the input CSV order changes.
    rng = np.random.default_rng(seed); chosen=[]
    quotas = {c:n//3+int(i<n%3) for i,c in enumerate(CATEGORIES)}
    for c in CATEGORIES:
        ids=sorted(frame.loc[frame.category==c,'query_id'])
        require(len(ids)>=quotas[c], 'Insufficient category '+c+'; no automatic redistribution')
        chosen.extend(rng.choice(ids, size=quotas[c], replace=False).tolist())
    return frame.set_index('query_id',drop=False).loc[sorted(chosen)].reset_index(drop=True)

def runtime_contract():
    return {'python':platform.python_version(), 'packages':{n:importlib.metadata.version(n) for n in
        ('torch','transformers','tokenizers','huggingface-hub','numpy','pandas','Pillow')},
        'device':str(DEVICE), 'device_name':torch.cuda.get_device_name() if DEVICE.type=='cuda' else platform.machine(),
        'cuda':torch.version.cuda, 'dtype':'float32', 'amp':False, 'tf32':False, 'attention_backend':'eager'}

def seed_all(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def rng_state():
    return {'python':random.getstate(),'numpy':np.random.get_state(),'torch':torch.get_rng_state(),
            'cuda':torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None}

def restore_rng(s):
    random.setstate(s['python']); np.random.set_state(s['numpy']); torch.set_rng_state(s['torch'])
    if s['cuda'] is not None:
        require(torch.cuda.is_available(), 'Resume CUDA checkpoint on CPU refused')
        torch.cuda.set_rng_state_all(s['cuda'])

def status(phase, **extra):
    write_json(RUN/'pilot_status.json', {'phase':phase,'implementation':IMPLEMENTATION,
        'official_val_used':False,'full_experiment_started':False,'updated_utc':pd.Timestamp.now(tz='UTC').isoformat(),**extra})

@contextlib.contextmanager
def stage(name):
    try: yield
    except BaseException as exc:
        if isinstance(exc, KeyboardInterrupt): kind='interrupted'
        else: kind='blocked'
        # No credential/download command logs are included in exception messages.
        write_bytes(RUN/(name+'_traceback.txt'), traceback.format_exc().encode())
        status(kind, stage=name, blocker=str(exc), traceback_file=name+'_traceback.txt')
        raise

def init_run():
    global DEVICE, RUN
    require(CFG['seed'] in (42,43,44) and CFG['amp'] is False, 'This version supports explicit FP32 only; AMP requires a new tested implementation')
    require(1<=CFG['pilot_epochs']<=3 and CFG['batch_size']>=2, 'Pilot limits: epochs 1..3, batch >=2')
    require(CFG['overfit_steps']>0 and CFG['checkpoint_every_steps']>0, 'Positive step budgets required')
    require(Path(CFG['run_id']).name==CFG['run_id'], 'run_id must be one path component')
    DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    torch.backends.cuda.matmul.allow_tf32=False; torch.backends.cudnn.allow_tf32=False
    torch.backends.cudnn.benchmark=False
    seed_all(CFG['seed'])
    root=Path(CFG['output_root']).resolve(); prep=Path(CFG['preparation_root']).resolve()
    RUN=root/CFG['run_id']
    for old in (prep, Path('/content/drive/MyDrive/ĐATN/data/processed/taskb_training/FashionIQ'),
                Path('/content/drive/MyDrive/ĐATN/data/processed/embeddings')):
        old=old.resolve(); require(RUN!=old and old not in RUN.parents and RUN not in old.parents, 'Pilot output overlaps existing assets')
    if RUN.exists() and any(RUN.iterdir()): require(CFG['resume'], 'Existing run requires resume=True or new run_id')
    RUN.mkdir(parents=True,exist_ok=True);Path(CFG['local_cache']).mkdir(parents=True,exist_ok=True)
    write_json(RUN/'pilot_config.json', {'config':CFG,'implementation':IMPLEMENTATION,'code_sha256':CODE_SHA256}, immutable=True)
    write_json(RUN/'runtime_contract.json', runtime_contract(), immutable=True)
    status('initialized', real_encoder_preflight=False, training_completed=False)
    print('Output riêng:', RUN, '| runtime:', DEVICE)


In [ ]:
def load_preparation():
    prep=Path(CFG['preparation_root'])
    for rel,sha in PREPARATION_ANCHORS.items():
        require(file_hash(prep/rel)==sha, 'Preparation changed: '+rel+'; inspect before updating anchors')
    protocol=read_json(prep/'protocol_config.json'); plan=read_json(prep/'training_plan.json')
    contract=read_json(prep/'extraction_contract.json'); meta=read_json(prep/'feature_metadata.json')
    require(contract==meta['contract'] and contract['caption_policy']==protocol['caption_policy']==CAPTION_POLICY, 'Extraction/caption contract mismatch')
    require(meta['status']=='complete' and not meta['legacy_arrays_mixed'], 'Incomplete/mixed feature package')
    feature_root=safe_child(prep,'features/'+meta['fingerprint'])
    indexes={}; mappings={}
    for kind in ('text','mask','reference','target'):
        p=feature_root/kind/'index.json'
        require(file_hash(p)==meta['store_index_sha256'][kind], 'Feature index checksum '+kind)
        idx=read_json(p); mapping=read_json(feature_root/meta['mapping_files'][kind])
        require(idx['fingerprint']==meta['fingerprint'] and idx['kind']==kind, 'Index identity mismatch')
        require(mapping=={v:i for i,v in enumerate(idx['ids'])} and len(mapping)==len(idx['ids']), 'ID mapping mismatch '+kind)
        indexes[kind]=idx; mappings[kind]=mapping
    frames={}; warnings=[]
    for name in ('development_train','development_val','development_gallery'):
        base=pd.read_csv(prep/(name+'.csv'),dtype=str,keep_default_na=False)
        view_record=meta['feature_views'][name]; path=safe_child(prep,view_record['path'])
        require(file_hash(path)==view_record['sha256'], 'Feature-view checksum '+name)
        view=pd.read_csv(path,dtype=str,keep_default_na=False)
        require(base.equals(view[base.columns]) and len(view)==view_record['rows'], 'Feature-view differs from source '+name)
        require(view_record['fingerprint']==meta['fingerprint'] and set(base.category)==set(CATEGORIES), 'View binding/category mismatch')
        if 'query_id' in base:
            require(base.query_id.is_unique and set(base.official_split)=={'train'}, 'Pilot queries must originate from official train only')
            for row in base.itertuples():
                require(row.query_id.startswith(row.category+':train:') and row.source_file=='cap.'+row.category+'.train.json', 'Query/source identity mismatch')
                require(row.query_id==f'{row.category}:train:{int(row.source_row):06d}', 'Source row/query ID mismatch')
                error,warning=assess_captions([row.caption_1,row.caption_2])
                require(error is None, 'Caption error '+row.query_id+': '+str(error))
                require(caption_text(row.caption_1,row.caption_2)==row.modification_text, 'Caption expression differs '+row.query_id)
                require(row.triplet_key_sha256==fingerprint([row.category,row.reference_image_id,row.target_image_id,row.caption_1,row.caption_2]), 'Triplet hash mismatch')
                require(all(v and v.replace('_','').replace('-','').isalnum() for v in (row.reference_image_id,row.target_image_id)), 'Invalid image ID')
                if warning: warnings.append({'query_id':row.query_id,'reason':warning})
            for col,kind,idcol in [('text_feature_index','text','query_id'),('reference_feature_index','reference','reference_image_id'),('target_feature_index','target','target_image_id')]:
                require(view[col].astype(int).tolist()==[mappings[kind][v] for v in view[idcol]], 'Feature mapping mismatch '+name+'/'+col)
        else:
            require(view.target_feature_index.astype(int).tolist()==[mappings['target'][v] for v in view.image_id], 'Gallery feature mapping mismatch')
            base.gallery_index=base.gallery_index.astype(int)
            for c in CATEGORIES:
                g=base[base.category==c]
                require(g.image_id.is_unique and g.gallery_index.tolist()==list(range(len(g))), 'Locked gallery ID order changed')
                require(fingerprint(g.image_id.tolist())==plan['development']['gallery_id_order_sha256'][c], 'Gallery lock hash mismatch')
        frames[name]=base
    train=frames['development_train']; val=frames['development_val']; gallery=frames['development_gallery']
    require(len(train)==plan['development']['train_count'] and len(val)==plan['development']['val_count'], 'Development counts differ from lock')
    require(set(train.query_id).isdisjoint(val.query_id), 'Train/val query overlap')
    require(set(train.target_image_id).isdisjoint(val.target_image_id), 'Validation targets leak into training supervision')
    for c in CATEGORIES:
        require(set(val.loc[val.category==c,'target_image_id'])<=set(gallery.loc[gallery.category==c,'image_id']), 'Validation target coverage missing')
    pilot_train=train.copy().reset_index(drop=True)
    require(len(train)==16204 and len(val)==1796,'Development lock count differs')
    overfit=stratified_sample(pilot_train,CFG['overfit_size'],CFG['seed'])
    pilot_val=val.copy().reset_index(drop=True)
    binding={'feature_fingerprint':meta['fingerprint'],'preparation_anchors':PREPARATION_ANCHORS,
             'feature_view_hashes':{n:meta['feature_views'][n]['sha256'] for n in frames},
             'mapping_hashes':{k:file_hash(feature_root/meta['mapping_files'][k]) for k in mappings}}
    selected={'overfit':overfit,'pilot_train':pilot_train,'pilot_val':pilot_val,'development_gallery':gallery}
    hashes={}
    for role,frame in selected.items():
        write_csv(RUN/'manifests'/(role+'.csv'),frame,immutable=True)
        hashes[role]={'sha256':file_hash(RUN/'manifests'/(role+'.csv')),'rows':len(frame),'counts':frame.groupby('category').size().to_dict()}
    write_json(RUN/'pilot_manifest_hashes.json',{'seed':CFG['seed'],'binding':binding,'manifests':hashes},immutable=True)
    write_json(RUN/'caption_warnings.json',{'warnings':warnings,'schema_errors':0,'both_empty_captions':0})
    write_json(RUN/'resource_inventory.json',{'preparation_root':str(prep),'binding':binding,
        'available_shapes':{k:v['shape_tail'] for k,v in indexes.items()},'available_counts':{k:len(v['ids']) for k,v in indexes.items()},
        'reuse':'pooled target/gallery only after weight/processor/content/feature integrity checks; stored final text/reference tokens are NOT intermediate layers',
        'official_val_read_or_evaluated':False},immutable=True)
    print(pd.DataFrame([{'role':k,'rows':v['rows'],**v['counts']} for k,v in hashes.items()]).to_string(index=False))
    return {'root':prep,'protocol':protocol,'plan':plan,'contract':contract,'meta':meta,'feature_root':feature_root,
            'indexes':indexes,'mappings':mappings,'frames':selected,'binding':binding,'manifest_hashes':hashes}

class PooledReader:
    def __init__(self, package):
        self.root=package['feature_root']/'target'; self.idx=package['indexes']['target']
        self.key=package['meta']['fingerprint']; self.mapping=package['mappings']['target']; self.loaded=OrderedDict()
        require(self.idx['shape_tail']==[512] and self.idx['dtype']=='float32', 'Pooled target schema differs')
    def take(self, ids):
        step=self.idx['shard_size']; out=[]
        for image_id in ids:
            row=self.mapping[image_id]; start=row//step*step; end=min(start+step,len(self.idx['ids']))
            if start not in self.loaded:
                name=f'{start:07d}_{end:07d}'; marker=read_json(self.root/(name+'.json')); path=self.root/(name+'.npy')
                expected={'fingerprint':self.key,'kind':'target','ids_sha256':fingerprint(self.idx['ids'][start:end]),'shape':[end-start,512],'dtype':'float32'}
                require(all(marker.get(k)==v for k,v in expected.items()) and file_hash(path)==marker['sha256'], 'Pooled shard checksum/schema/ID mismatch')
                a=np.load(path,allow_pickle=False); require(a.shape==(end-start,512) and a.dtype==np.float32 and np.isfinite(a).all(),'Invalid pooled shard')
                require(np.allclose(np.linalg.norm(a,axis=-1),1,atol=1e-5,rtol=1e-5),'Nonunit pooled targets')
                if len(self.loaded)>=8:self.loaded.popitem(last=False)
                self.loaded[start]=a
            self.loaded.move_to_end(start); out.append(self.loaded[start][row-start].copy())
        return np.stack(out)

def find_image_directory(required_ids):
    # Inspect existing runtime/Drive data first; no annotation from official-val is read.
    candidates=[]
    for root in map(Path,CFG['image_search_roots']):
        if not root.is_dir(): continue
        if any((root/(v+'.jpg')).is_file() for v in required_ids[:3]): candidates.append(root)
        for current,dirs,files in os.walk(root):
            dirs[:]=[d for d in dirs if d not in ('features','embeddings','taskb_training','benchmark_preparation','tme_pilot','.git','__MACOSX') and not d.startswith('.')]
            if any(v+'.jpg' in files for v in required_ids[:3]): candidates.append(Path(current))
    full=sorted(set(p.resolve() for p in candidates if all((p/(v+'.jpg')).is_file() for v in required_ids)))
    require(len(full)<=1,'Several image directories have full coverage; set CFG.image_search_roots explicitly: '+str(full))
    return full[0] if full else None

def kaggle_command(args,timeout=120):
    try:
        r=subprocess.run([shutil.which('kaggle'),*args],capture_output=True,text=True,timeout=timeout,stdin=subprocess.DEVNULL)
    except Exception: raise RuntimeError('Kaggle command failed before completion; check runtime/network. Raw logs suppressed.') from None
    if r.returncode==0:return 'ok'
    text=(r.stdout+'\n'+r.stderr).lower()
    if any(x in text for x in ('401','403','unauthor','forbidden','credential','authentication','kaggle.json','api_token')):return 'auth_required'
    raise RuntimeError('Kaggle download/listing failed (exit '+str(r.returncode)+'); raw logs suppressed to protect credentials')

def upload_kaggle_credentials():
    from google.colab import files
    private=Path('/root/.kaggle'); private.mkdir(mode=0o700,exist_ok=True)
    with tempfile.TemporaryDirectory(dir=private) as tmp:
        cwd=Path.cwd(); uploaded={}
        try:
            os.chdir(tmp); uploaded=files.upload()
            require(len(uploaded)==1,'Upload exactly one Kaggle credential JSON')
            name=next(iter(uploaded)); payload=json.loads(uploaded[name].decode('utf-8'))
            require(isinstance(payload,dict) and all(isinstance(payload.get(k),str) and payload[k].strip() for k in ('username','key')), 'Kaggle JSON requires username and key')
            fd,temp=tempfile.mkstemp(dir=private)
            with os.fdopen(fd,'w') as f:
                os.fchmod(f.fileno(),0o600); json.dump({k:payload[k].strip() for k in ('username','key')},f)
            os.replace(temp,private/'kaggle.json'); (private/'kaggle.json').chmod(0o600)
            # Uploaded JSON is explicitly selected; avoid stale environment auth taking precedence.
            for k in ('KAGGLE_API_TOKEN','KAGGLE_USERNAME','KAGGLE_KEY'):os.environ.pop(k,None)
            os.environ['KAGGLE_CONFIG_DIR']=str(private)
        except Exception: raise RuntimeError('Credential upload/validation failed; no credential content recorded') from None
        finally:os.chdir(cwd); uploaded.clear()

def extract_safely(archive,root):
    root=Path(root).resolve()
    with zipfile.ZipFile(archive) as bundle:
        for member in bundle.infolist():
            p=PurePosixPath(member.filename)
            require(not p.is_absolute() and '..' not in p.parts and '\\' not in member.filename and not stat.S_ISLNK(member.external_attr>>16),'Unsafe ZIP member')
            if any(x.startswith('._') or x=='__MACOSX' for x in p.parts): continue
            dest=safe_child(root,str(p))
            if member.is_dir():dest.mkdir(parents=True,exist_ok=True);continue
            dest.parent.mkdir(parents=True,exist_ok=True)
            if dest.exists():
                h=hashlib.sha256()
                with bundle.open(member) as src:
                    for block in iter(lambda:src.read(8*1024*1024),b''):h.update(block)
                require(dest.is_file() and file_hash(dest)==h.hexdigest(),'Existing dataset file differs; do not overwrite: '+str(dest));continue
            temp=dest.with_name(dest.name+'.'+uuid.uuid4().hex+'.extracting')
            with bundle.open(member) as src,temp.open('wb') as dst:shutil.copyfileobj(src,dst)
            os.replace(temp,dest)

def prepare_images(package, required_ids):
    directory=find_image_directory(required_ids)
    if directory is None:
        require(CFG['allow_kaggle_download'],'Images incomplete in existing roots; enable Kaggle download explicitly')
        require(shutil.which('kaggle') is not None,'Run bootstrap first to install Kaggle CLI')
        root=Path(CFG['dataset_root']); root.mkdir(parents=True,exist_ok=True)
        if kaggle_command(['datasets','files','-d','binhhuunguyen/fashioniq','--page-size','1'])=='auth_required':upload_kaggle_credentials()
        with tempfile.TemporaryDirectory(dir=root,prefix='.download-') as tmp:
            args=['datasets','download','-d','binhhuunguyen/fashioniq','-p',tmp,'--quiet']
            outcome=kaggle_command(args,timeout=3600)
            if outcome=='auth_required':upload_kaggle_credentials();outcome=kaggle_command(args,timeout=3600)
            require(outcome=='ok','Kaggle authentication denied; do not continue')
            archives=list(Path(tmp).glob('*.zip')); require(len(archives)==1 and zipfile.is_zipfile(archives[0]),'Dataset ZIP incomplete')
            extract_safely(archives[0],root)
        directory=find_image_directory(required_ids)
    require(directory is not None,'Required images still missing after preparation')
    # This is a full audit of every image used by this pilot, not all FashionIQ images.
    original=read_json(package['root']/'image_content_manifest.json')
    audit=read_json(package['root']/'data_audit.json')
    require(audit['passed'] and not audit['partial_mode'] and fingerprint(original)==audit['image_content_order_sha256'],'Preparation image manifest/audit mismatch')
    expected={r['image_id']:r['sha256'] for r in original}
    require(len(expected)==len(original),'Preparation image IDs duplicated')
    rows=[]; bad=[]
    for image_id in required_ids:
        p=directory/(image_id+'.jpg')
        try:
            require(p.is_file() and not p.name.startswith('._') and not p.is_symlink(),'Missing/unsafe image')
            with Image.open(p) as im:require(im.format=='JPEG','Not JPEG'); im.verify()
            with Image.open(p) as im:im.load(); size=list(im.size)
            sha=file_hash(p); require(sha==expected[image_id],'Image differs from preparation bytes')
            rows.append({'image_id':image_id,'sha256':sha,'size':size})
        except Exception as exc:bad.append({'image_id':image_id,'reason':str(exc)})
    write_json(RUN/'pilot_image_audit.json',{'scope':'all images required by pilot only','images_dir':str(directory),'images':rows,'errors':bad,'missing_or_corrupt':len(bad),'passed':not bad})
    require(not bad,'Pilot image audit failed; see pilot_image_audit.json; no zero tensors or query removal')
    return directory,{r['image_id']:r['sha256'] for r in rows}


In [ ]:
class ReferenceInjection(nn.Module):
    def __init__(self, text_dim, vision_dim, heads, mlp):
        super().__init__()
        self.q_ln=nn.LayerNorm(text_dim); self.v_ln=nn.LayerNorm(vision_dim)
        self.v_proj=nn.Linear(vision_dim,text_dim)
        self.ca=nn.MultiheadAttention(text_dim,heads,batch_first=True,dropout=0.0)
        self.out_ln=nn.LayerNorm(text_dim); self.mlp=copy.deepcopy(mlp)
    def forward(self, h, reference):
        v=self.v_proj(self.v_ln(reference))
        z=h+self.ca(self.q_ln(h),v,v,need_weights=False)[0]
        return z+self.mlp(self.out_ln(z))

def eos_positions(ids, mask, eos_token_id):
    require(ids.shape==mask.shape and bool(((mask==0)|(mask==1)).all()), 'Token/mask shape or values invalid')
    lengths=mask.sum(-1)
    require(bool((lengths>=2).all()), 'Empty text token sequence')
    expected=torch.arange(ids.shape[1],device=ids.device)[None,:]<lengths[:,None]
    require(torch.equal(expected,mask.bool()), 'Tokenizer must right-pad contiguous sequences')
    # pad_token == EOS for CLIP; intersect with attention_mask and select first real EOS.
    is_eos=(ids==eos_token_id)&mask.bool()
    require(bool(is_eos.any(-1).all()), 'EOS absent from unpadded tokens')
    pos=is_eos.long().argmax(-1)
    require(bool((pos==lengths-1).all()), 'EOS must close each untruncated modifier')
    return pos

def text_masks(mask, dtype):
    b,n=mask.shape; neg=torch.finfo(dtype).min
    padding=torch.zeros((b,1,n,n),device=mask.device,dtype=dtype).masked_fill(~mask[:,None,None,:].bool(),neg)
    causal=torch.full((n,n),neg,dtype=dtype,device=mask.device).triu(1)
    return padding,causal[None,None].expand(b,1,n,n)

class TMEPilot(nn.Module):
    """Paper-inspired layer-level tri-branch; not the unpublished original TME."""
    def __init__(self, base, eos_token_id, active_layers):
        super().__init__()
        self.text=copy.deepcopy(base.text_model)
        self.projection=copy.deepcopy(base.text_projection)
        self.text.requires_grad_(True);self.projection.requires_grad_(True)
        self.eos_token_id=int(eos_token_id); self.active_layers=tuple(active_layers)
        self.align=nn.ModuleDict(); self.inject=nn.ModuleDict(); self.merge=nn.ModuleDict()
        self.align_norm=nn.ModuleDict(); self.align_projection=nn.ModuleDict()
        self.text.embeddings.requires_grad_(False)  # pilot choice: freeze token/position tables
        tc=base.config.text_config; vc=base.config.vision_config
        require(len(self.text.encoder.layers)==vc.num_hidden_layers, 'This implementation requires paired text/vision depths')
        require(self.active_layers and len(set(self.active_layers))==len(self.active_layers) and
                all(0<=i<len(self.text.encoder.layers) for i in self.active_layers),'Invalid active layers')
        for i in self.active_layers:
            k=str(i); block=self.text.encoder.layers[i]
            self.align[k]=copy.deepcopy(block)
            self.inject[k]=ReferenceInjection(tc.hidden_size,vc.hidden_size,tc.num_attention_heads,block.mlp)
            self.merge[k]=nn.LayerNorm(tc.hidden_size)
            self.align_norm[k]=copy.deepcopy(self.text.final_layer_norm)
            self.align_projection[k]=copy.deepcopy(base.text_projection)
            self.align_projection[k].requires_grad_(True)

    def forward(self, reference, modifier, return_alignment=False, conventional_only=False, return_sequence=False):
        # reference: layer_tokens[B,L,Nv,Dv], pooled[B,D]; contains NO target.
        ids=modifier['input_ids']; mask=modifier['attention_mask']
        pos=eos_positions(ids,mask,self.eos_token_id); b=ids.shape[0]
        h=self.text.embeddings(input_ids=ids)
        padding,causal=text_masks(mask,h.dtype); aligned=[]
        valid=mask.unsqueeze(-1).to(h.dtype)
        for i,block in enumerate(self.text.encoder.layers):
            previous=h
            conventional=block(previous,attention_mask=padding,causal_attention_mask=causal)[0]
            if i in self.active_layers and not conventional_only:
                k=str(i)
                ri=self.inject[k](previous,reference['layer_tokens'][:,self.active_layers.index(i)])
                ta=self.align[k](previous,attention_mask=padding,causal_attention_mask=causal)[0]
                # Equation 4: enhanced stream is propagated into every subsequent layer.
                h=self.merge[k](conventional+ri+ta)*valid
                a=self.align_norm[k](ta)[torch.arange(b,device=h.device),pos]
                aligned.append(self.align_projection[k](a))
            else: h=conventional*valid
        h=self.text.final_layer_norm(h)
        if return_sequence: return h*valid, mask
        pooled=h[torch.arange(b,device=h.device),pos]
        text=unit(self.projection(pooled))
        q=unit(text+unit(reference['pooled']))
        if return_alignment:
            require(aligned, 'No TA outputs in conventional-only path')
            return q, torch.stack(aligned,dim=1)
        return q

    def encode_query(self, reference, modifier):
        return self(reference, modifier)  # deliberately no target argument

def positive_mask(target_ids, device):
    require(len(target_ids)>0 and all(isinstance(v,str) and v for v in target_ids), 'Invalid target IDs')
    return torch.tensor([[a==b for b in target_ids] for a in target_ids],dtype=torch.bool,device=device)

def retrieval_loss(query, teacher, target_ids, temperature):
    require(temperature>0, 'Invalid retrieval temperature')
    q=unit(query); t=unit(teacher.detach()); require(q.shape==t.shape, 'Retrieval dimensions differ')
    logits=q@t.T/temperature; positives=positive_mask(target_ids,q.device)
    def direction(scores,p):
        return -((F.log_softmax(scores,dim=1)*p).sum(1)/p.sum(1)).mean()
    return (direction(logits,positives)+direction(logits.T,positives.T))/2

def alignment_loss(student, teacher, temperature):
    # Paper KL(z_align || z_target): forward KL from student P to detached teacher Q.
    # [B,L,D] -> probability across D; sum over D, mean over B then L, times T^2.
    require(student.shape==teacher.shape and student.ndim==3 and temperature>0, 'KL requires matching [B,L,D]')
    s=unit(student); t=unit(teacher.detach())
    log_p=F.log_softmax(s/temperature,dim=-1)
    log_q=F.log_softmax(t/temperature,dim=-1)
    return (log_p.exp()*(log_p-log_q)).sum(-1).mean()*temperature**2

def losses(model, batch):
    query, aligned=model(batch['reference'],batch['modifier'],return_alignment=True)
    retrieval=retrieval_loss(query,batch['target_pooled'],batch['target_ids'],CFG['retrieval_temperature'])
    alignment=alignment_loss(aligned,batch['target_layers'],CFG['alignment_temperature'])
    total=retrieval+CFG['alignment_weight']*alignment
    finite(total,'total loss'); return total,retrieval,alignment

def gradient_norms(model):
    groups={'conventional':list(model.text.encoder.parameters())+list(model.projection.parameters()),
            'reference_injection':list(model.inject.parameters()),'target_alignment':list(model.align.parameters())+
                list(model.align_norm.parameters())+list(model.align_projection.parameters()),'merge':list(model.merge.parameters())}
    result={}
    for name,params in groups.items():
        gs=[p.grad for p in params if p.grad is not None]
        require(gs and all(bool(torch.isfinite(g).all()) for g in gs), 'Missing/nonfinite gradient '+name)
        result[name]=float(torch.sqrt(sum(g.detach().float().square().sum() for g in gs)))
    return result


In [ ]:
def load_locked_encoder(package):
    require(not CFG['require_cuda'] or DEVICE.type=='cuda','Select a GPU runtime in Colab before loading/encoding/training this pilot')
    c=package['contract']
    require(c['encoder_id']=='patrickjohncyh/fashion-clip' and c['revision']=='7e3ba62ce16b379a1ab479346b66f192e76f51b7','Unexpected encoder contract')
    tokenizer=CLIPTokenizer.from_pretrained(c['encoder_id'],revision=c['revision'])
    processor=CLIPImageProcessor.from_pretrained(c['encoder_id'],revision=c['revision'])
    require(type(tokenizer).__name__==c['tokenizer']['class'] and fingerprint(tokenizer.get_vocab())==c['tokenizer']['vocab_sha256'],'Tokenizer differs from lock')
    require(tokenizer.special_tokens_map==c['tokenizer']['special_tokens'] and tokenizer.padding_side=='right','Special tokens/padding mismatch')
    actual=processor.to_dict()
    require(all(actual.get(k)==v for k,v in c['processor'].items() if k not in ('_processor_class','image_processor_type')),'Preprocessing differs from locked contract')
    base=CLIPModel.from_pretrained(c['encoder_id'],revision=c['revision'],dtype=torch.float32,attn_implementation='eager')
    require(tensor_hash(base)==c['encoder_weight_sha256'],'Loaded encoder weight hash differs from preparation')
    base.requires_grad_(False); base.eval(); base.to(DEVICE)
    tc=base.config.text_config; vc=base.config.vision_config
    require((tc.hidden_size,vc.hidden_size,base.config.projection_dim)==(512,768,512),'Unexpected Fashion-CLIP dimensions')
    active=list(range(tc.num_hidden_layers)) if CFG['active_layers'] is None else CFG['active_layers']
    require(active==sorted(active),'List active_layers in increasing order')
    label='full-depth pilot' if active==list(range(tc.num_hidden_layers)) else 'reduced-layer prototype'
    model=TMEPilot(base,tokenizer.eos_token_id,active).to(DEVICE)
    architecture={'implementation':IMPLEMENTATION,'label':label,'exact_original_reproduction':False,
        'sources':SOURCE_REFERENCES,'active_layers_zero_based':active,'text_depth':tc.num_hidden_layers,
        'vision_depth':vc.num_hidden_layers,'dimensions':{'vision':768,'text':512,'retrieval':512},
        'flow':'At layer l: same h(l-1) -> conventional CLIP block, RI cross-attention to vision layer l, independent TA CLIP block; LN(conv+RI+TA) -> h(l). All encoder layers still execute in reduced-layer prototype.',
        'teacher':'Frozen vision CLS after each corresponding block -> frozen final vision post-LN + visual_projection -> L2; stop gradient. No target enters query branches.',
        'pooling':'First real EOS from tokenizer IDs intersected with attention_mask, final text LN -> pretrained trainable text projection -> L2; query=L2(L2(text)+L2(reference pooled)).',
        'masks':'Right padding; 4D additive key-padding + causal upper-triangular mask in conventional/TA; padded outputs zeroed before next layer; RI attends only to unpadded vision tokens.',
        'pilot_choices_not_fully_specified_in_paper':['Independent pretrained copy for each TA block; token/position embeddings frozen, conventional blocks/final text LN/projection trainable.',
            'RI pre-LN query/key-value, learned vision 768->512 per layer, CLIP-MLP copied initialization, dropout 0.',
            'TA EOS projected to retrieval space at each selected layer; teacher uses final vision post-LN/projection at every layer.',
            'KL over embedding channels (not token positions); forward KL P_student||Q_teacher; layer mean rather than paper sum; L2 vectors before softmax; explicit T^2 scaling.',
            'Retrieval symmetric multi-positive mean log-probability; fixed temperature; equal normalized ref+text composition; no XAA or DVR.'],
        'loss':{'retrieval':'symmetric multi-positive contrastive; mask based on target image ID; mean over positives per row, then rows and directions',
            'retrieval_temperature':CFG['retrieval_temperature'],'alignment':'P=softmax(L2(TA_EOS)/T), Q=softmax(detach(L2(vision_CLS_projected))/T); sum_D P*(logP-logQ); mean_B_and_L; multiply T^2',
            'alignment_shapes':'both [B,selected_layers,512]','alignment_temperature':CFG['alignment_temperature'],'alignment_weight':CFG['alignment_weight'],'total':'retrieval + weight*alignment'},
        'trainable_parameters':sum(p.numel() for p in model.parameters() if p.requires_grad),
        'frozen_vision_parameters':sum(p.numel() for p in base.vision_model.parameters()),'caption_policy':CAPTION_POLICY}
    write_json(RUN/'architecture_contract.json',architecture,immutable=True)
    rows=[['Vision + visual_projection','B,3,224,224 -> L x B,50,768 / B,512','frozen','reference layers + detached teacher'],
          ['Text token/position tables','B,77 -> B,77,512','frozen','pretrained tokenization/positions'],
          ['Conventional CLIP blocks','B,77,512 -> B,77,512','trainable','causal/padding-aware text path'],
          ['RI per active layer','text + B,50,768 -> B,77,512','trainable','768->512 cross-attention, residual+LN/MLP'],
          ['TA per active layer','B,77,512 -> B,77,512 / B,512','trainable','separate causal block + EOS alignment'],
          ['Merge / next layer','3 x B,77,512 -> B,77,512','trainable LN','enhanced stream to next layer'],
          ['Final pooling/projection','EOS B,512 -> query B,512','trainable','L2(text+reference); no target']]
    print(pd.DataFrame(rows,columns=['module','input/output shape','state','role']).to_string(index=False))
    print(label,'| trainable:',f"{architecture['trainable_parameters']:,}")
    return base,tokenizer,processor,model,architecture

def tokenize_rows(frame, tokenizer):
    values=frame.modification_text.tolist()
    lengths=[len(x) for x in tokenizer(values,padding=False,truncation=False)['input_ids']]
    bad=[{'query_id':q,'tokens':n} for q,n in zip(frame.query_id,lengths) if n>77]
    require(not bad,'Overlength modifiers: '+str(bad[:10])+'; do not truncate/drop queries')
    return tokenizer(values,padding='max_length',max_length=77,truncation=False,return_tensors='pt')

@torch.no_grad()
def vision_outputs(base,pixels,active):
    base.eval(); out=base.vision_model(pixel_values=pixels,output_hidden_states=True)
    # hidden_states[0] is embedding/pre-LN input; i+1 is actual output of vision block i.
    tokens=torch.stack([out.hidden_states[i+1] for i in active],dim=1)
    teacher=unit(base.visual_projection(base.vision_model.post_layernorm(tokens[:,:,0,:])))
    pooled=unit(base.visual_projection(out.pooler_output))
    finite(tokens,'vision layer tokens'); require(tokens.dtype==torch.float32,'Vision cache must be FP32')
    return tokens.detach(), teacher.detach(), pooled.detach()

class PilotCache:
    def __init__(self, root, contract, ids_by_role, shapes):
        self.key=fingerprint(contract); self.root=Path(root)/self.key
        self.local=Path(CFG['local_cache'])/self.key; self.local.mkdir(parents=True,exist_ok=True)
        self.contract=contract; self.ids_by_role=ids_by_role; self.shapes=shapes
        self.maps={k:{v:i for i,v in enumerate(ids)} for k,ids in ids_by_role.items()}
        require(all(len(self.maps[k])==len(ids) for k,ids in ids_by_role.items()),'Cache IDs duplicated')
        self.root.mkdir(parents=True,exist_ok=True); self.loaded=OrderedDict(); self.verified={}
        write_json(self.root/'index.json',{'fingerprint':self.key,'contract':contract,'ids':ids_by_role,'shapes':shapes,'dtype':'float32','shard_size':CFG['cache_shard_size']},immutable=True)
    def paths(self,role,start):
        end=min(start+CFG['cache_shard_size'],len(self.ids_by_role[role]));name=f'{start:07d}_{end:07d}'
        p=self.root/role; p.mkdir(exist_ok=True)
        expected={'fingerprint':self.key,'role':role,'ids_sha256':fingerprint(self.ids_by_role[role][start:end]),'shape':[end-start,*self.shapes[role]],'dtype':'float32'}
        return p/(name+'.npy'),p/(name+'.json'),expected
    def read(self,role,start,compute=None):
        source,marker,expected=self.paths(role,start); pending=marker.with_suffix('.pending.json')
        if not marker.exists():
            if pending.exists():
                m=read_json(pending)
                require(all(m.get(k)==v for k,v in expected.items()),'Pending cache contract differs')
                if source.exists():
                    require(file_hash(source)==m['sha256'],'Pending cache checksum mismatch')
                    write_json(marker,m,immutable=True)
            if not marker.exists():
                require(not source.exists(),'Orphan cache has no provenance; preserve and investigate')
                require(compute is not None,'Cache shard not committed')
                ids=self.ids_by_role[role][start:start+CFG['cache_shard_size']]
                values=np.asarray(compute(ids),dtype=np.float32)
                require(list(values.shape)==expected['shape'] and np.isfinite(values).all(),'Computed shard shape/finite error')
                if role!='reference_layers':require(np.allclose(np.linalg.norm(values,axis=-1),1,atol=1e-5,rtol=1e-5),'Cache vectors not normalized')
                with tempfile.NamedTemporaryFile(dir=self.local,suffix='.npy',delete=False) as f:np.save(f,values,allow_pickle=False);tmp=Path(f.name)
                m={**expected,'sha256':file_hash(tmp)}
                write_json(pending,m)  # transaction commit recoverable after an interrupted copy
                stagepath=source.with_suffix('.copying');shutil.copyfile(tmp,stagepath)
                require(file_hash(stagepath)==m['sha256'],'Cache copy checksum differs')
                os.replace(stagepath,source);write_json(marker,m,immutable=True);tmp.unlink()
        m=read_json(marker)
        require(all(m.get(k)==v for k,v in expected.items()),'Cache fingerprint/schema/ID order mismatch')
        key=(role,start); local=self.local/role/source.name;local.parent.mkdir(exist_ok=True)
        if key not in self.verified:
            require(file_hash(source)==m['sha256'],'Drive cache checksum mismatch')
            if not local.exists() or file_hash(local)!=m['sha256']:
                temp=local.with_suffix('.copying');shutil.copyfile(source,temp);os.replace(temp,local)
            a=np.load(local,mmap_mode='r',allow_pickle=False)
            require(list(a.shape)==m['shape'] and a.dtype==np.float32 and np.isfinite(a).all(),'Cache contents invalid')
            if role!='reference_layers':require(np.allclose(np.linalg.norm(a,axis=-1),1,atol=1e-5,rtol=1e-5),'Committed cache contains nonunit vectors')
            self.verified[key]=(local.stat().st_size,local.stat().st_mtime_ns)
        require((local.stat().st_size,local.stat().st_mtime_ns)==self.verified[key],'Local cache changed; rerun cache preparation')
        if key not in self.loaded:
            if len(self.loaded)>=8:self.loaded.popitem(last=False)
            self.loaded[key]=np.load(local,mmap_mode='r',allow_pickle=False)
        self.loaded.move_to_end(key);return self.loaded[key]
    def ensure(self,role,compute):
        for start in range(0,len(self.ids_by_role[role]),CFG['cache_shard_size']):self.read(role,start,compute)
        print(role,':',len(self.ids_by_role[role]),'IDs verified')
    def take(self,role,ids):
        rows=[self.maps[role][v] for v in ids]; step=CFG['cache_shard_size']
        return np.stack([np.asarray(self.read(role,r//step*step)[r%step]).copy() for r in rows])

def prepare_pilot_cache(package,base,processor,model):
    frames=package['frames']; refs=sorted(set(frames['pilot_train'].reference_image_id)|set(frames['pilot_val'].reference_image_id))
    targets=sorted(set(frames['pilot_train'].target_image_id))
    pooled_ids=sorted(set(targets)|set(frames['development_gallery'].image_id))
    # Full gallery images audited even when pooled features are reused.
    all_ids=sorted(set(refs)|set(pooled_ids))
    directory,image_hashes=prepare_images(package,all_ids)
    layer_ids=list(model.active_layers)
    def encode_fresh(ids, role):
        result=[]
        for start in range(0,len(ids),CFG['encode_batch_size']):
            batch=ids[start:start+CFG['encode_batch_size']]; images=[]
            for image_id in batch:
                p=directory/(image_id+'.jpg');require(file_hash(p)==image_hashes[image_id],'Image changed after audit')
                with Image.open(p) as im:images.append(im.convert('RGB'))
            pixels=processor(images=images,return_tensors='pt')['pixel_values'].float().to(DEVICE)
            tokens,teacher,pooled=vision_outputs(base,pixels,layer_ids)
            result.append({'reference_layers':tokens,'target_layers':teacher,'pooled':pooled}[role].cpu().numpy())
        return np.concatenate(result)
    old_cache=None
    old_root=Path(CFG['pilot_root'])
    if (old_root/'cache_binding.json').is_file():
        old_binding=read_json(old_root/'cache_binding.json');old_path=old_root/'cache'/old_binding['fingerprint']/'index.json'
        require(file_hash(old_path)==old_binding['index_sha256'],'Pilot index checksum differs')
        old_index=read_json(old_path);oc=old_index['contract']
        compatible=(oc['encoder_revision']==package['contract']['revision'] and oc['weight_sha256']==package['contract']['encoder_weight_sha256']
          and oc['preprocessor']==package['contract']['processor'] and oc['layers']==layer_ids and oc['dtype']=='float32'
          and oc['runtime']==runtime_contract() and oc['caption_policy']==CAPTION_POLICY)
        if compatible:
            old_cache=PilotCache(old_root/'cache',oc,old_index['ids'],old_index['shapes'])
            require(old_cache.key==old_binding['fingerprint'],'Pilot cache fingerprint differs')
    reuse_counts={}
    def encode(ids,role):
        reusable=[v for v in ids if old_cache is not None and v in old_cache.maps[role]
                  and old_cache.contract['image_hashes'].get(v)==image_hashes[v]]
        fresh_ids=[v for v in ids if v not in set(reusable)]
        values={}
        if reusable:values.update(zip(reusable,old_cache.take(role,reusable)))
        if fresh_ids:values.update(zip(fresh_ids,encode_fresh(fresh_ids,role)))
        reuse_counts[role]=reuse_counts.get(role,0)+len(reusable)
        return np.stack([values[v] for v in ids])
    reader=PooledReader(package)
    probe=pooled_ids[:min(8,len(pooled_ids))]
    old=reader.take(probe); fresh=encode_fresh(probe,'pooled')
    comparison={'ids':probe,'max_abs':float(np.abs(old-fresh).max()),'mean_abs':float(np.abs(old-fresh).mean()),
                'min_cosine':float(np.sum(old*fresh,axis=-1).min()),'atol':CFG['pooled_reuse_atol'],'rtol':0.0,
                'scope':'sample numerical compatibility only; all used old shards separately checksum/finite/norm verified'}
    reuse=bool(CFG['reuse_verified_pooled'] and np.allclose(old,fresh,atol=CFG['pooled_reuse_atol'],rtol=0.0))
    comparison['reused']=reuse
    comparison['fallback']='none' if reuse else 'reencode only pilot targets + locked development gallery; never change threshold'
    write_json(RUN/'pooled_reuse_diagnostic.json',comparison)
    contract={'implementation':IMPLEMENTATION,'code_sha256':CODE_SHA256,'preparation':package['binding'],
        'encoder_revision':package['contract']['revision'],'weight_sha256':package['contract']['encoder_weight_sha256'],
        'preprocessor':package['contract']['processor'],'layers':layer_ids,'layer_indexing':'hidden_states[i+1], physical output of vision block i',
        'image_hashes':image_hashes,'id_orders':{'reference':refs,'target_teacher':targets,'pooled':pooled_ids},
        'runtime':runtime_contract(),'pooled_source':'verified preparation target store' if reuse else 'fresh frozen pilot/gallery encode',
        'pooled_reuse_diagnostic':comparison,'dtype':'float32','caption_policy':CAPTION_POLICY}
    cache=PilotCache(Path(CFG['output_root'])/'shared_cache',contract,{'reference_layers':refs,'target_layers':targets,'reference_pooled':refs,'pooled':pooled_ids},
        {'reference_layers':[len(layer_ids),50,768],'target_layers':[len(layer_ids),512],'reference_pooled':[512],'pooled':[512]})
    cache.ensure('reference_layers',lambda ids:encode(ids,'reference_layers'))
    # Reference pooled features are computed from genuine final vision states, not averaged intermediate states.
    cache.ensure('reference_pooled',lambda ids:encode(ids,'pooled'))
    cache.ensure('target_layers',lambda ids:encode(ids,'target_layers'))
    cache.ensure('pooled',reader.take if reuse else lambda ids:encode(ids,'pooled'))
    write_json(RUN/'cache_binding.json',{'fingerprint':cache.key,'index_sha256':file_hash(cache.root/'index.json'),'all_required_shards_verified':True},immutable=True)
    write_json(RUN/'pilot_cache_reuse.json',reuse_counts)
    return cache

def make_batch(frame,cache,tokenizer,training=True):
    tokens=tokenize_rows(frame,tokenizer)
    reference={'layer_tokens':torch.from_numpy(cache.take('reference_layers',frame.reference_image_id.tolist())).to(DEVICE),
               'pooled':torch.from_numpy(cache.take('reference_pooled',frame.reference_image_id.tolist())).to(DEVICE)}
    batch={'reference':reference,'modifier':{k:v.to(DEVICE) for k,v in tokens.items()},'query_ids':frame.query_id.tolist()}
    if training:
        # Validation calls training=False: no target tensor can enter inference.
        batch.update(target_pooled=torch.from_numpy(cache.take('pooled',frame.target_image_id.tolist())).to(DEVICE),
                     target_layers=torch.from_numpy(cache.take('target_layers',frame.target_image_id.tolist())).to(DEVICE),target_ids=frame.target_image_id.tolist())
    return batch


In [ ]:
def exact_ranks(query,gallery,target_positions):
    q=np.asarray(query,dtype=np.float32); g=np.asarray(gallery,dtype=np.float32)
    require(q.ndim==g.ndim==2 and q.shape[1]==g.shape[1] and len(q)>0 and len(g)>0,'Ranking shapes invalid')
    require(np.isfinite(q).all() and np.isfinite(g).all(),'Nonfinite ranking input')
    require((np.linalg.norm(q,axis=-1)>1e-8).all() and (np.linalg.norm(g,axis=-1)>1e-8).all(),'Zero ranking input')
    q=q/np.linalg.norm(q,axis=-1,keepdims=True);g=g/np.linalg.norm(g,axis=-1,keepdims=True)
    targets=np.asarray(target_positions,dtype=np.int64)
    require(targets.shape==(len(q),) and (targets>=0).all() and (targets<len(g)).all(),'Target gallery alignment invalid')
    scores=q@g.T; ts=scores[np.arange(len(q)),targets]
    ranks=1+(scores>ts[:,None]).sum(1)+((scores==ts[:,None])&(np.arange(len(g))[None,:]<targets[:,None])).sum(1)
    return ranks

def macro_metrics(per_category):
    require(set(per_category)==set(CATEGORIES),'Macro requires three categories')
    overall={f'R@{k}':float(np.mean([per_category[c][f'R@{k}'] for c in CATEGORIES])) for k in (1,5,10,50)}
    return {'overall':overall,'Mean':(overall['R@10']+overall['R@50'])/2}

def pure_tests():
    cases=[(['red','long'],None,None,'Red and long'),(['','long'],None,'single_empty_caption',' and long'),
           (['red',''],None,'single_empty_caption','Red and '),([' .?, ','long'],None,'single_empty_caption',' and long'),
           ([' \t ','.?, '],'both_empty_captions',None,None),(['',''],'both_empty_captions',None,None),
           (['red',1],'schema_error',None,None),(['red'],'schema_error',None,None),('red','schema_error',None,None)]
    for caps,error,warning,text in cases:
        require(assess_captions(caps)==(error,warning),'Caption assessment mismatch')
        if text is not None:require(caption_text(*caps)==text,'Caption policy mismatch')
    require(caption_text(' .Red?, ',' BLUE. ')=='Red and BLUE','Caption normalization strips only .?, and ASCII space at both ends')
    require(normalize_caption('\tblue\t')=='\tblue\t','Tabs must not be silently stripped by policy')
    q=np.array([[1,0],[0,1]],np.float32);g=np.array([[1,0],[1,0],[0,1]],np.float32)
    require(exact_ranks(q,g,[1,2]).tolist()==[2,1],'Manual rank/tie test failed')
    rng=np.random.default_rng(42);a=rng.normal(size=(7,8)).astype(np.float32);b=rng.normal(size=(13,8)).astype(np.float32);targets=np.arange(7)
    a/=np.linalg.norm(a,axis=-1,keepdims=True);b/=np.linalg.norm(b,axis=-1,keepdims=True)
    orders=np.argsort(-(a@b.T),axis=1,kind='stable')
    expected=[int(np.flatnonzero(row==t)[0])+1 for row,t in zip(orders,targets)]
    require(exact_ranks(a,b,targets).tolist()==expected,'Exact rank and stable sorting disagree')
    cat={c:{'R@1':0.,'R@5':10.,'R@10':20.+i*10,'R@50':60.+i*10} for i,c in enumerate(CATEGORIES)}
    require(macro_metrics(cat)['overall']['R@10']==30 and macro_metrics(cat)['Mean']==50,'Macro/Mean test failed')
    ids=['a','a','b','c'];mask=positive_mask(ids,torch.device('cpu'));require(mask[0,1] and not mask[0,2],'Duplicate target positive mask failed')
    x=unit(torch.arange(1,17,dtype=torch.float32).reshape(4,4));y=x.flip(0)
    logits=x@y.T/0.07
    manual=[]
    for scores,p in ((logits,mask),(logits.T,mask.T)):
        manual.append(torch.stack([-F.log_softmax(scores[i],dim=0)[p[i]].mean() for i in range(4)]).mean())
    require(torch.allclose(retrieval_loss(x,y,ids,.07),sum(manual)/2),'Multi-positive reduction mismatch')
    student=torch.randn(2,2,4,requires_grad=True);teacher=torch.randn(2,2,4,requires_grad=True)
    value=alignment_loss(student,teacher,.07);value.backward()
    require(teacher.grad is None and student.grad is not None and torch.isfinite(student.grad).all(),'Alignment teacher detach failed')
    p=F.softmax(unit(student.detach())/.07,dim=-1);lp=p.log();lq=F.log_softmax(unit(teacher.detach())/.07,dim=-1)
    require(torch.allclose(value.detach(),(p*(lp-lq)).sum(-1).mean()*.07**2,atol=1e-7),'KL direction/channel/reduction mismatch')
    for bad in (torch.randn(2,3,4),torch.randn(2,2,5)):
        try:alignment_loss(student,bad,.07)
        except RuntimeError:pass
        else:raise AssertionError('KL accepted wrong shape')
    return {'caption_cases':len(cases)+2,'exact_rank_tie_and_stable_sort':True,'macro_mean':True,
            'duplicate_target_multi_positive':True,'alignment_direction_shape_detach':True}

def model_tests(base,model,modifier,reference,target_layers,target_pooled,scope):
    saved_rng=rng_state();model.eval();base.eval()
    vision_before=tensor_hash(base.vision_model);projection_before=tensor_hash(base.visual_projection)
    with tempfile.TemporaryDirectory() as temp:
        initial=Path(temp)/'initial.pt';torch.save(model.state_dict(),initial)
        try:
            with torch.no_grad():
                q=model.encode_query(reference,modifier)
                require(q.shape==(len(modifier['input_ids']),base.config.projection_dim),'Query shape failed')
                require(q.dtype==torch.float32 and torch.isfinite(q).all() and torch.allclose(q.norm(dim=-1),torch.ones(len(q),device=q.device),atol=1e-5),'Query dtype/finite/unit failed')
                q1=model.encode_query({k:v[:1] for k,v in reference.items()},{k:v[:1] for k,v in modifier.items()})
                require(torch.allclose(q1,q[:1],atol=1e-5,rtol=1e-5),'Batch-size 1 differs from normal batch')
                native=base.text_model(**modifier).pooler_output
                expected=unit(unit(base.text_projection(native))+unit(reference['pooled']))
                conventional=model(reference,modifier,conventional_only=True)
                require(torch.allclose(expected,conventional,atol=1e-5,rtol=1e-5),'Conventional adapter does not match native CLIP text forward')
                mutated={k:v.clone() for k,v in modifier.items()}
                mutated['input_ids'][~mutated['attention_mask'].bool()]=(model.eos_token_id+3)%base.config.text_config.vocab_size
                require(torch.allclose(q,model.encode_query(reference,mutated),atol=1e-5,rtol=1e-5),'Padding token content changes embedding')
                pad,causal=text_masks(modifier['attention_mask'],torch.float32)
                require(bool((causal[0,0].triu(1)[torch.triu(torch.ones_like(causal[0,0]),diagonal=1).bool()]<0).all()),'Causal mask absent')
                require(bool((pad.masked_select(~modifier['attention_mask'][:,None,None,:].expand_as(pad).bool())<0).all()),'Padding key mask absent')
                # Changing target tensors can change losses, never query encoding; target not accepted by API.
                import inspect
                require(list(inspect.signature(model.encode_query).parameters)==['reference','modifier'],'Inference API accepts target')
                changed_teacher=target_layers.flip(0).clone()
                alignment_loss(torch.randn_like(changed_teacher),changed_teacher,CFG['alignment_temperature'])
                require(torch.equal(q,model.encode_query(reference,modifier)),'Target-dependent query embedding')
            model.train();opt=torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],lr=CFG['learning_rate'],weight_decay=CFG['weight_decay'])
            target_ids=['same' if i<2 else str(i) for i in range(len(q))]
            teacher=target_layers.detach().clone().requires_grad_(True)
            pooled=target_pooled.detach().clone().requires_grad_(True)
            batch={'reference':reference,'modifier':modifier,'target_layers':teacher,'target_pooled':pooled,'target_ids':target_ids}
            total,retrieval,align=losses(model,batch);total.backward();norms=gradient_norms(model)
            require(all(v>0 and math.isfinite(v) for v in norms.values()),'A trainable branch has no gradient')
            require(teacher.grad is None and pooled.grad is None,'Target teacher received gradient')
            require(all(p.grad is not None and bool(torch.isfinite(p.grad).all()) for p in model.parameters() if p.requires_grad),'Some trainable parameters are disconnected')
            require(all(p.grad is None for p in base.parameters()),'Frozen encoder received gradient')
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad],CFG['gradient_clip'],error_if_nonfinite=True)
            opt.step();require(all(bool(torch.isfinite(p).all()) for p in model.parameters()),'Step produced nonfinite weights')
            require(tensor_hash(base.vision_model)==vision_before and tensor_hash(base.visual_projection)==projection_before,'Frozen vision weights changed')
            del opt
            result={'scope':scope,'passed':True,'native_conventional_parity':True,'batch_1_and_normal':True,'batch_sizes_tested':[1,len(q)],'padding_and_causal_masks':True,
                'finite_normalized_fp32':True,'gradient_norms':norms,'all_trainable_gradients_finite':True,'vision_unchanged_and_no_gradient':True,
                'target_teacher_detached':True,'inference_independent_of_target':True,'one_optimizer_step':True,
                'initial_loss':{'retrieval':float(retrieval.detach()),'alignment':float(align.detach()),'total':float(total.detach())}}
        finally:
            model.zero_grad(set_to_none=True)
            model.load_state_dict(torch.load(initial,map_location='cpu',weights_only=True),strict=True);model.eval();restore_rng(saved_rng)
        with torch.no_grad():restored=model.encode_query(reference,modifier)
        require(torch.equal(q,restored),'Save/load failed to restore output exactly on the same runtime')
        result['save_load_output_restored']=True
    return result

@contextlib.contextmanager
def resource_measure(label):
    if DEVICE.type=='cuda':torch.cuda.synchronize();torch.cuda.reset_peak_memory_stats()
    start=time.perf_counter()
    try:yield
    finally:
        if DEVICE.type=='cuda':torch.cuda.synchronize()
        p=RUN/'resource_profile.json';report=read_json(p) if p.exists() else {'runtime':runtime_contract(),'measurements':[]}
        report['measurements'].append({'label':label,'elapsed_seconds':time.perf_counter()-start,
            'cuda_peak_allocated_bytes':torch.cuda.max_memory_allocated() if DEVICE.type=='cuda' else None,
            'cuda_peak_reserved_bytes':torch.cuda.max_memory_reserved() if DEVICE.type=='cuda' else None,
            'gpu_measured':DEVICE.type=='cuda'})
        write_json(p,report)

def real_preflight(base,tokenizer,processor,model):
    report={'scope':'real locked Fashion-CLIP, synthetic RGB pixel inputs and real tokenizer/model; not a dataset audit','passed':False}
    try:
        require(not CFG['require_cuda'] or DEVICE.type=='cuda','GPU required for this training pilot. In Colab choose Runtime > Change runtime type > GPU, then run from the top.')
        with resource_measure('real_encoder_preflight_forward_backward_step'):
            examples=['Red and longer',' and blue','Short sleeves and ']
            texts=(examples*math.ceil(CFG['batch_size']/len(examples)))[:CFG['batch_size']]
            mod=tokenizer(texts,padding='max_length',max_length=77,truncation=False,return_tensors='pt')
            mod={k:v.to(DEVICE) for k,v in mod.items()}
            grid=np.arange(224*224*3,dtype=np.uint32).reshape(224,224,3)
            images=[Image.fromarray(((grid+i*47)%256).astype(np.uint8)) for i in range(len(texts))]
            pixels=processor(images=images,return_tensors='pt')['pixel_values'].to(DEVICE,dtype=torch.float32)
            tokens,teachers,pooled=vision_outputs(base,pixels,list(model.active_layers))
            report.update(model_tests(base,model,mod,{'layer_tokens':tokens,'pooled':pooled},teachers,pooled,'real locked pretrained encoder on runtime'))
            report['pure_tests']=pure_tests();report['runtime']=runtime_contract();report['passed']=True
    except BaseException:
        report['traceback']=traceback.format_exc();raise
    finally:write_json(RUN/'test_report.json',report)
    status('preflight_passed',real_encoder_preflight=True,training_completed=False)
    print('Preflight encoder thật đạt; trạng thái và phép đo đã lưu. Chưa train pilot.')
    return report

@torch.no_grad()
def evaluate_pilot(model,package,cache,tokenizer):
    model.eval();queries=package['frames']['pilot_val'];gallery=package['frames']['development_gallery'];by_category={};rows=[]
    for c in CATEGORIES:
        q=queries[queries.category==c];g=gallery[gallery.category==c].sort_values('gallery_index',kind='stable')
        require(q.query_id.is_unique and g.image_id.is_unique and g.gallery_index.tolist()==list(range(len(g))),'Evaluation ID/order error')
        ids=g.image_id.tolist();lookup={v:i for i,v in enumerate(ids)};vectors=cache.take('pooled',ids);ranks=[]
        for start in range(0,len(q),CFG['eval_batch_size']):
            batch_rows=q.iloc[start:start+CFG['eval_batch_size']]
            batch=make_batch(batch_rows,cache,tokenizer,training=False)
            require('target_layers' not in batch and 'target_pooled' not in batch,'Validation target entered query')
            value=model.encode_query(batch['reference'],batch['modifier']).float().cpu().numpy()
            positions=[lookup[v] for v in batch_rows.target_image_id]
            r=exact_ranks(value,vectors,positions);ranks.extend(r.tolist())
            rows.extend({'query_id':source.query_id,'category':c,'reference_image_id':source.reference_image_id,'target_image_id':source.target_image_id,'target_rank':int(rank)} for source,rank in zip(batch_rows.itertuples(),r))
        by_category[c]={f'R@{k}':float((np.asarray(ranks)<=k).mean()*100) for k in (1,5,10,50)}
    require(len(rows)==len(queries) and {r['query_id'] for r in rows}==set(queries.query_id),'Lost validation queries')
    metrics={'scope':'full locked development validation; NOT official-val','unit':'percent','per_category':by_category,**macro_metrics(by_category),
        'query_counts':queries.groupby('category').size().to_dict(),'gallery_counts':gallery.groupby('category').size().to_dict(),
        'search':'exact L2 cosine FP32, reference retained, exact ties original gallery_index ascending','n_query':len(queries)}
    return metrics,pd.DataFrame(rows)


In [ ]:
def checkpoint_save(path,state,signature):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    require(state['signature']==signature,'Checkpoint signature differs')
    with tempfile.NamedTemporaryFile(dir=CFG['local_cache'],suffix='.pt',delete=False) as f:temp=Path(f.name)
    try:
        torch.save(state,temp);meta={'signature':signature,'sha256':file_hash(temp),'phase':state.get('phase'),'step':state.get('step')}
        write_json(path.with_suffix('.pending.json'),meta)
        stagepath=path.with_suffix('.copying');shutil.copyfile(temp,stagepath)
        require(file_hash(stagepath)==meta['sha256'],'Checkpoint copy checksum failed')
        os.replace(stagepath,path);write_json(path.with_suffix('.json'),meta)
    finally:
        if temp.exists():temp.unlink()

def checkpoint_load(path,signature):
    path=Path(path);marker=path.with_suffix('.json');pending=path.with_suffix('.pending.json')
    require(path.is_file(),'Checkpoint not found')
    sha=file_hash(path)
    if pending.is_file():
        p=read_json(pending)
        if sha==p.get('sha256'):
            require(p['signature']==signature,'Pending checkpoint config mismatch');write_json(marker,p)
    require(marker.is_file(),'Checkpoint has no integrity marker; preserve and investigate')
    meta=read_json(marker);require(meta['signature']==signature and meta['sha256']==sha,'Checkpoint checksum/config mismatch')
    # Only this pilot's checksum/contract-bound self-created checkpoint, never arbitrary downloaded pickle.
    state=torch.load(path,map_location='cpu',weights_only=False)
    require(state['signature']==signature,'Checkpoint payload signature mismatch')
    return state

def run_binding(package,cache,architecture):
    return {'config':CFG,'code_sha256':CODE_SHA256,'implementation':IMPLEMENTATION,'preparation':package['binding'],
        'manifests':package['manifest_hashes'],'cache_fingerprint':cache.key,'architecture':architecture,'runtime':runtime_contract()}

def ensure_initial(model,signature):
    path=RUN/'initial_model.pt'
    if path.exists():
        state=checkpoint_load(path,signature)
    else:
        state={'signature':signature,'phase':'initial','step':0,'model':model.state_dict()}
        checkpoint_save(path,state,signature)
    return path

def batches_for_epoch(frame,batch_size,seed,epoch):
    ids=np.asarray(sorted(frame.query_id),dtype=object)
    require(len(ids)>=2 and batch_size>=2,'Contrastive training requires >=2 examples')
    order=np.random.default_rng(seed+epoch).permutation(ids)
    batches=[order[i:i+batch_size].tolist() for i in range(0,len(ids),batch_size)]
    # Preserve every query: merge a singleton tail into the preceding batch, never drop it.
    if len(batches)>1 and len(batches[-1])==1:batches[-2].extend(batches.pop())
    return batches

@torch.no_grad()
def overfit_probe(model,frame,cache,tokenizer):
    model.eval();qs=[];ts=[];aligned=[];teachers=[]
    for start in range(0,len(frame),CFG['batch_size']):
        rows=frame.iloc[start:start+CFG['batch_size']];batch=make_batch(rows,cache,tokenizer,True)
        q,a=model(batch['reference'],batch['modifier'],return_alignment=True)
        qs.append(q);ts.append(batch['target_pooled']);aligned.append(a);teachers.append(batch['target_layers'])
    q=torch.cat(qs);t=torch.cat(ts);ids=frame.target_image_id.tolist()
    retrieval=float(retrieval_loss(q,t,ids,CFG['retrieval_temperature']))
    alignment=float(alignment_loss(torch.cat(aligned),torch.cat(teachers),CFG['alignment_temperature']))
    # Training-only diagnostic: correct if top-1 target ID equals any duplicate positive ID.
    top=(q@unit(t).T).argmax(-1).cpu().tolist()
    r1=float(np.mean([ids[j]==ids[i] for i,j in enumerate(top)])*100)
    return {'retrieval':retrieval,'alignment':alignment,'total':retrieval+CFG['alignment_weight']*alignment,'R@1_train_bank':r1,'samples':len(frame)}

def derived_training_outputs(state):
    write_csv(RUN/(state['phase']+'_training_history.csv'),pd.DataFrame(state['history']))
    if state['phase']=='overfit':
        write_json(RUN/'overfit_diagnostic.json',{'before':state['before'],'after':state.get('after'),
            'learning_signal':state.get('learning_signal'),'finished':state['finished'],
            'criterion':'fixed full overfit bank retrieval loss relative drop >= CFG.overfit_min_relative_drop OR positive change in training-bank R@1; no requirement of monotonic step loss'})
    else:
        write_json(RUN/'pilot_validation_metrics.json',{'scope':'fixed development only; no benchmark/SOTA claim',
            'before':state['before'],'epochs':state['validation'],'finished':state['finished'],'selected_best_checkpoint':False})
    # A single requested history file, both phases; never concatenate unrelated run IDs.
    tables=[]
    for phase in ('overfit','pilot'):
        p=RUN/(phase+'_training_history.csv')
        if p.exists():tables.append(pd.read_csv(p))
    if tables:write_csv(RUN/'training_history.csv',pd.concat(tables,ignore_index=True))



In [ ]:
def infrastructure_tests():
    # Small synthetic I/O fixtures validate storage logic, not real encoder/training quality.
    saved_rng=rng_state()
    with tempfile.TemporaryDirectory() as temp:
        root=Path(temp); contract={'fixture':True,'revision':'unit','layers':[0,1],'id_order':['a','b']}
        a=np.eye(2,dtype=np.float32)
        cache=PilotCache(root/'cache',contract,{'pooled':['a','b']},{'pooled':[2]})
        cache.ensure('pooled',lambda ids:a[[{'a':0,'b':1}[v] for v in ids]])
        require(np.array_equal(cache.take('pooled',['b','a']),a[[1,0]]),'Cache ID mapping failed')
        same=PilotCache(root/'cache',contract,{'pooled':['a','b']},{'pooled':[2]})
        require(np.array_equal(same.take('pooled',['a','b']),a),'Cache resume failed')
        other=PilotCache(root/'cache',{**contract,'layers':[1]}, {'pooled':['a','b']},{'pooled':[2]})
        require(cache.key!=other.key,'Layer change failed to invalidate cache fingerprint')
        source,marker,expected=cache.paths('pooled',0); raw=source.read_bytes()
        source.write_bytes(raw[:-1]+bytes([raw[-1]^1]))
        try: PilotCache(root/'cache',contract,{'pooled':['a','b']},{'pooled':[2]}).read('pooled',0)
        except RuntimeError:pass
        else:raise AssertionError('Corrupt cache was accepted')
        source.write_bytes(raw)
        # Recover committed bytes if interrupted after data rename and before commit marker.
        m=read_json(marker);marker.unlink();write_json(marker.with_suffix('.pending.json'),m)
        recovered=PilotCache(root/'cache',contract,{'pooled':['a','b']},{'pooled':[2]})
        require(np.array_equal(recovered.take('pooled',['a','b']),a),'Pending cache recovery failed')
        seed_all(42);linear=nn.Linear(3,2);opt=torch.optim.AdamW(linear.parameters(),lr=.01)
        inp=torch.randn(4,3)
        def step():
            opt.zero_grad(set_to_none=True); loss=linear(inp).square().mean(); loss.backward(); opt.step()
        step();signature=fingerprint({'fixture':42});path=root/'checkpoint.pt'
        checkpoint_save(path,{'signature':signature,'phase':'test','step':1,'model':linear.state_dict(),'optimizer':opt.state_dict(),'rng':rng_state()},signature)
        step();expected_model=copy.deepcopy(linear.state_dict()); expected_rand=torch.rand(3)
        saved=checkpoint_load(path,signature);linear.load_state_dict(saved['model']);opt.load_state_dict(saved['optimizer']);restore_rng(saved['rng'])
        step()
        require(all(torch.equal(linear.state_dict()[k],v) for k,v in expected_model.items()) and torch.equal(torch.rand(3),expected_rand),'Optimizer/RNG resume differs from uninterrupted step')
        try:checkpoint_load(path,fingerprint({'fixture':43}))
        except RuntimeError:pass
        else:raise AssertionError('Wrong-config checkpoint accepted')
        marker=path.with_suffix('.json');marker.unlink()
        require(checkpoint_load(path,signature)['step']==1,'Pending checkpoint recovery failed')
        # All rows included even with singleton tail; batches use persistent query IDs.
        fixture=pd.DataFrame({'query_id':[f'q{i}' for i in range(9)]})
        batches=batches_for_epoch(fixture,4,42,0)
        require(sorted(sum(batches,[]))==sorted(fixture.query_id) and all(len(v)>=2 for v in batches),'Singleton tail dropped query')
    restore_rng(saved_rng)
    return {'scope':'synthetic storage/optimizer fixtures','cache_id_order_resume':True,'cache_layer_fingerprint':True,
            'corruption_rejected':True,'pending_transaction_recovery':True,'optimizer_rng_resume':True,'wrong_signature_rejected':True,'no_dropped_singleton_query':True}


In [ ]:
class DevelopmentModel(TMEPilot):
    def __init__(self,base,eos,active,variant):
        super().__init__(base,eos,active);self.variant=variant
        require(variant in ('conventional','no_alignment','full'),'Unknown ablation')
        if variant=='conventional':
            for group in (self.inject,self.align,self.merge,self.align_norm,self.align_projection):group.requires_grad_(False)
        elif variant=='no_alignment':
            # TA still contributes to the enhanced stream; its auxiliary projection heads have no active objective.
            self.align_norm.requires_grad_(False);self.align_projection.requires_grad_(False)
    def forward(self,reference,modifier,return_alignment=False,conventional_only=False,return_sequence=False):
        return super().forward(reference,modifier,return_alignment=return_alignment,
            conventional_only=conventional_only or self.variant=='conventional',return_sequence=return_sequence)
    def enhanced_text_sequence(self,reference,modifier):
        return self(reference,modifier,return_sequence=True)

def losses(model,batch):
    if getattr(model,'variant','full')=='conventional':
        query=model.encode_query(batch['reference'],batch['modifier']);alignment=query.new_zeros(())
    else:
        query,aligned=model(batch['reference'],batch['modifier'],return_alignment=True)
        alignment=alignment_loss(aligned,batch['target_layers'],CFG['alignment_temperature'])
    retrieval=retrieval_loss(query,batch['target_pooled'],batch['target_ids'],CFG['retrieval_temperature'])
    weight=CFG['alignment_weight'] if getattr(model,'variant','full')=='full' else 0.
    total=retrieval+weight*alignment
    finite(total,'total');return total,retrieval,alignment

def gradient_norms(model):
    groups={'conventional':[model.text.encoder,model.text.final_layer_norm,model.projection],
            'reference_injection':[model.inject],'target_alignment':[model.align,model.align_norm,model.align_projection],'merge':[model.merge]}
    result={}
    # One host transfer for group norms, rather than one .item() per parameter.
    tensors=[];names=[]
    for name,modules in groups.items():
        params=[p for m in modules for p in m.parameters() if p.requires_grad]
        if not params:continue
        grads=[p.grad for p in params if p.grad is not None]
        require(grads,'No gradient in active group '+name)
        tensors.append(torch.stack([g.detach().float().square().sum() for g in grads]).sum().sqrt());names.append(name)
    values=torch.stack(tensors).detach().cpu().tolist()
    require(all(math.isfinite(v) for v in values),'Nonfinite branch gradient')
    return dict(zip(names,values))

_TOKEN_ROWS={}
if '_original_tokenize_rows' not in globals():_original_tokenize_rows=tokenize_rows
def tokenize_rows(frame,tokenizer):
    ids=frame.query_id.tolist()
    if all(q in _TOKEN_ROWS for q in ids):
        return {k:torch.stack([_TOKEN_ROWS[q][k] for q in ids]) for k in ('input_ids','attention_mask')}
    return _original_tokenize_rows(frame,tokenizer)

def prime_tokens(package,tokenizer):
    for role in ('pilot_train','pilot_val'):
        f=package['frames'][role]
        for start in range(0,len(f),256):
            rows=f.iloc[start:start+256];tokens=_original_tokenize_rows(rows,tokenizer)
            for i,q in enumerate(rows.query_id):_TOKEN_ROWS[q]={k:v[i].clone() for k,v in tokens.items()}

class LocalTrainingCache:
    """All metadata/checksums validated once before training; batch access touches local mmap only."""
    def __init__(self,source):
        self.key=source.key;self.contract=source.contract;self.maps=source.maps;self.paths={};self.loaded=OrderedDict()
        bytes_needed=sum(len(source.ids_by_role[k])*math.prod(source.shapes[k])*4 for k in source.shapes)
        require(shutil.disk_usage(CFG['local_cache']).free>4*1024**3,'Local disk insufficient for verified training cache + checkpoints')
        for role,ids in source.ids_by_role.items():
            for start in range(0,len(ids),CFG['cache_shard_size']):
                source.read(role,start) # full checksum, finite/norm and ID/schema checks outside train
                p=source.local/role/source.paths(role,start)[0].name
                self.paths[role,start]=p
        write_json(RUN/'local_cache_ready.json',{'fingerprint':self.key,'bytes':bytes_needed,'verified_shards':len(self.paths),'per_batch_hashing':False})
    def take(self,role,ids):
        output=[]
        for image_id in ids:
            row=self.maps[role][image_id];start=row//CFG['cache_shard_size']*CFG['cache_shard_size'];key=(role,start)
            if key not in self.loaded:
                if len(self.loaded)>=16:self.loaded.popitem(last=False)
                self.loaded[key]=np.load(self.paths[key],mmap_mode='r',allow_pickle=False)
            self.loaded.move_to_end(key);output.append(np.asarray(self.loaded[key][row-start]).copy())
        return np.stack(output)

def cache_resource_plan(package):
    f=package['frames'];refs=set(f['pilot_train'].reference_image_id)|set(f['pilot_val'].reference_image_id)
    targets=set(f['pilot_train'].target_image_id);pooled=targets|set(f['development_gallery'].image_id)
    sizes={'reference_layers':len(refs)*12*50*768*4,'target_layers':len(targets)*12*512*4,
           'reference_pooled':len(refs)*512*4,'pooled':len(pooled)*512*4}
    from google.colab import auth
    auth.authenticate_user()
    import google.auth
    from googleapiclient.discovery import build
    credentials,_=google.auth.default()
    quota=build('drive','v3',credentials=credentials,cache_discovery=False).about().get(fields='storageQuota').execute()['storageQuota']
    disk=shutil.disk_usage('/content');free_vram,total_vram=torch.cuda.mem_get_info() if DEVICE.type=='cuda' else (0,0)
    import psutil
    report={'counts':{'reference':len(refs),'teacher':len(targets),'pooled':len(pooled)},'bytes':sizes,'total_bytes':sum(sizes.values()),
       'local_disk_free':disk.free,'ram_available':psutil.virtual_memory().available,'vram_free':free_vram,'vram_total':total_vram,
       'strategy':'FP32 full-depth shards on Drive; complete validated local mirror before training; stop if insufficient. No layer/precision reduction.',
       'local_additional_reserve_bytes':12*1024**3,'drive_quota':'Verify available Drive storage for total_bytes plus 3 variants x best/last checkpoint (~10 GB). Drive quota cannot be inferred from FUSE disk_usage.'}
    existing_local=0;existing_drive=0
    binding_path=RUN/'cache_binding.json'
    if binding_path.is_file():
        binding=read_json(binding_path);index_path=Path(CFG['output_root'])/'shared_cache'/binding['fingerprint']/'index.json'
        if index_path.is_file() and file_hash(index_path)==binding['index_sha256']:
            index=read_json(index_path)
            if index['contract']['code_sha256']==CODE_SHA256 and index['contract']['preparation']==package['binding'] and index['contract']['runtime']==runtime_contract():
                if binding.get('all_required_shards_verified'):existing_drive=sum(sizes.values())
                for role,ids in index['ids'].items():
                    for start in range(0,len(ids),CFG['cache_shard_size']):
                        end=min(start+CFG['cache_shard_size'],len(ids));p=Path(CFG['local_cache'])/binding['fingerprint']/role/f'{start:07d}_{end:07d}.npy'
                        if p.is_file():existing_local+=min(p.stat().st_size,(end-start)*math.prod(index['shapes'][role])*4)
    report['reusable_allocated_local_bytes']=existing_local
    report['reusable_committed_drive_bytes']=existing_drive
    report['drive_storage_quota']=quota
    write_json(RUN/'cache_resource_plan.json',report)
    require('limit' in quota,'Drive did not return a bounded quota; verify storage before enabling full cache')
    require(int(quota['limit'])-int(quota.get('usage',0))>sum(sizes.values())-existing_drive+10*1024**3,'Drive quota insufficient for cache plus best/last checkpoints')
    write_json(RUN/'cache_resource_plan.json',report);print(json.dumps(report,indent=2))
    require(disk.free>sum(sizes.values())-existing_local+12*1024**3,'Insufficient local disk; choose larger disk runtime before encoding. No automatic precision/layer reduction')
    return report

def initialization_for_variant(base,tokenizer,variant):
    saved=rng_state();seed_all(CFG['seed'])
    model=DevelopmentModel(base,tokenizer.eos_token_id,list(range(base.config.text_config.num_hidden_layers)),variant).to(DEVICE)
    restore_rng(saved);return model

def alignment_gradient_probe(model,batch):
    total,r,a=losses(model,batch)
    params=[p for p in model.parameters() if p.requires_grad]
    gr=torch.autograd.grad(r,params,retain_graph=True,allow_unused=True)
    weight=CFG['alignment_weight'] if model.variant=='full' else 0.
    ga=torch.autograd.grad(weight*a,params,allow_unused=True) if weight else [None]*len(params)
    device=r.device
    nr=sum((g.detach().float().square().sum() for g in gr if g is not None),torch.zeros((),device=device))
    na=sum((g.detach().float().square().sum() for g in ga if g is not None),torch.zeros((),device=device))
    dot=sum(((x.detach().float()*y.detach().float()).sum() for x,y in zip(gr,ga) if x is not None and y is not None),torch.zeros((),device=device))
    vals=torch.stack([nr.sqrt(),na.sqrt(),dot/(nr.sqrt()*na.sqrt()).clamp_min(1e-30)]).cpu().tolist()
    return dict(zip(('retrieval_gradient_norm','weighted_alignment_gradient_norm','gradient_cosine'),vals))

def variant_preflight(model,base,batch):
    saved=rng_state();model.eval();before=tensor_hash(model);vision=tensor_hash(base.vision_model)
    with tempfile.TemporaryDirectory() as tmp:
        initial=Path(tmp)/'model.pt';torch.save(model.state_dict(),initial)
        try:
            for n in (1,len(batch['query_ids'])):
                ref={k:v[:n] for k,v in batch['reference'].items()};mod={k:v[:n] for k,v in batch['modifier'].items()}
                q=model.encode_query(ref,mod);require(q.shape==(n,base.config.projection_dim),'Query shape')
                sequence,mask=model.enhanced_text_sequence(ref,mod)
                require(sequence.shape[:2]==mask.shape and sequence.shape[-1]==base.config.text_config.hidden_size,'Enhanced sequence schema')
                finite(sequence,'sequence');finite(q,'query')
            with torch.no_grad():
                q0=model.encode_query(batch['reference'],batch['modifier']).clone()
                changed_supervision={**batch,'target_layers':batch['target_layers'].flip(0),'target_pooled':batch['target_pooled'].flip(0)}
                require(torch.equal(q0,model.encode_query(changed_supervision['reference'],changed_supervision['modifier'])),'Changed target altered query')
                native=unit(base.text_projection(base.text_model(**batch['modifier']).pooler_output))
                conventional=super(DevelopmentModel,model).forward(batch['reference'],batch['modifier'],conventional_only=True)
                expected=unit(native+unit(batch['reference']['pooled']))
                require(torch.allclose(conventional,expected,atol=3e-5,rtol=3e-5),'Native/adapter mismatch')
            teacher=batch['target_layers'].detach().clone().requires_grad_();target=batch['target_pooled'].detach().clone().requires_grad_()
            test_batch={**batch,'target_layers':teacher,'target_pooled':target}
            opt=torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],lr=CFG['learning_rate'],weight_decay=CFG['weight_decay'])
            t0=time.perf_counter();total,r,a=losses(model,test_batch);total.backward();norms=gradient_norms(model)
            require(all(p.grad is not None and torch.isfinite(p.grad).all() for p in model.parameters() if p.requires_grad),'Active parameter disconnected/nonfinite')
            require(teacher.grad is None and target.grad is None and all(p.grad is None for p in base.parameters()),'Teacher/vision gradient')
            torch.nn.utils.clip_grad_norm_(model.parameters(),CFG['gradient_clip'],error_if_nonfinite=True);opt.step()
            require(all(torch.isfinite(p).all() for p in model.parameters()),'Optimizer produced nonfinite weights')
            if DEVICE.type=='cuda':torch.cuda.synchronize()
            elapsed=time.perf_counter()-t0
            probe=alignment_gradient_probe(model,batch)
            # Query API never receives target; changing supervision cannot enter inference.
            model.load_state_dict(torch.load(initial,map_location='cpu',weights_only=True),strict=True)
            model.eval()
            with torch.no_grad():q1=model.encode_query(batch['reference'],batch['modifier'])
            require(torch.equal(q0,q1),'Temporary step/save-load did not restore query')
            require(tensor_hash(base.vision_model)==vision and all(not p.requires_grad for p in base.parameters()),'Vision changed')
            report={'passed':True,'variant':model.variant,'batch_sizes':[1,len(batch['query_ids'])],'branch_gradients':norms,'alignment_influence':probe,
             'native_adapter':True,'teacher_detached':True,'target_independent_query':True,'save_load':True,
             'trainable_parameters':sum(p.numel() for p in model.parameters() if p.requires_grad),
             'sample_step_seconds':elapsed,'estimated_epoch_compute_seconds':elapsed*math.ceil(16204/CFG['batch_size']),
             'timing_limit':'one real batch; excludes full epoch I/O, evaluation and checkpoint synchronization'}
        finally:
            model.load_state_dict(torch.load(initial,map_location='cpu',weights_only=True),strict=True);model.zero_grad(set_to_none=True);restore_rng(saved)
    require(tensor_hash(model)==before,'Preflight changed initialization');return report

def development_train(variant):
    global RUN
    RUN=ROOT_RUN/variant;RUN.mkdir(exist_ok=True)
    model=initialization_for_variant(BASE,TOKENIZER,variant)
    signature=fingerprint({'variant':variant,'config':CFG,'code':CODE_SHA256,'cache':CACHE.key,'preparation':PACKAGE['binding'],'architecture':ARCHITECTURE})
    write_json(RUN/'config.json',{'signature':signature,'variant':variant,'config':CFG,'initial_sha256':tensor_hash(model),
       'initialization':'new locked pretrained, same seed for shared components; no pilot/overfit weights',
       'scheduler':None,'scaler':None,'checkpoint_policy':'local atomic first; Drive at each epoch and every 1024 steps; loss up to 1023 steps if local runtime disappears'},immutable=True)
    batch=make_batch(PACKAGE['frames']['pilot_train'].iloc[:CFG['batch_size']],CACHE,TOKENIZER)
    try:
        report=variant_preflight(model,BASE,batch)
        write_json(RUN/'test_report.json',{'passed':[report],'failed':[],'skipped':[]})
    except BaseException:
        write_json(RUN/'test_report.json',{'passed':[],'failed':[traceback.format_exc()],'skipped':['training blocked']})
        raise
    del batch
    opt=torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],lr=CFG['learning_rate'],weight_decay=CFG['weight_decay'],betas=(.9,.999),eps=1e-8)
    local=Path('/content/visfit_tme_development_checkpoints')/CFG['run_id']/variant;local.mkdir(parents=True,exist_ok=True)
    state={'signature':signature,'phase':variant,'step':0,'epoch':0,'cursor':0,'history':[],'metrics':[],'best_mean':None,'best_epoch':None,'bad_epochs':0,'finished':False}
    drive_last=RUN/'last.pt';local_last=local/'last.pt'
    candidates=[]
    for path in (local_last,drive_last):
        if path.exists():
            candidate=checkpoint_load(path,signature);candidates.append(candidate)
    if candidates:
        state=max(candidates,key=lambda s:s['step']);model.load_state_dict(state['model'],strict=True);opt.load_state_dict(state['optimizer']);restore_rng(state['rng'])
    else:seed_all(CFG['seed'])
    def commit(best=False):
        state.update(model=model.state_dict(),optimizer=opt.state_dict(),scheduler=None,scaler=None,rng=rng_state(),config=CFG,cache_fingerprint=CACHE.key)
        checkpoint_save(local_last,state,signature)
        # Copy committed bytes, preserving checksum/marker; no checkpoint hash in per-batch loop.
        for name in (['last','best'] if best else ['last']):
            dest=RUN/(name+'.pt');temp=dest.with_suffix('.copying');shutil.copyfile(local_last,temp)
            marker=read_json(local_last.with_suffix('.json'));write_json(dest.with_suffix('.pending.json'),marker);require(file_hash(temp)==marker['sha256'],'Checkpoint Drive copy corrupt')
            os.replace(temp,dest);write_json(dest.with_suffix('.json'),marker)
        write_csv(RUN/'training_history.csv',pd.DataFrame(state['history']));write_json(RUN/'metric_history.json',state['metrics'])
    try:
        for epoch in range(state['epoch'],CFG['max_epochs']):
            if state['finished']:break
            order=batches_for_epoch(PACKAGE['frames']['pilot_train'],CFG['batch_size'],CFG['seed'],epoch)
            indexed=PACKAGE['frames']['pilot_train'].set_index('query_id',drop=False)
            start=time.perf_counter()
            if DEVICE.type=='cuda':torch.cuda.reset_peak_memory_stats()
            for cursor in range(state['cursor'],len(order)):
                rows=indexed.loc[order[cursor]];batch=make_batch(rows,CACHE,TOKENIZER)
                model.train();opt.zero_grad(set_to_none=True)
                total,r,a=losses(model,batch);total.backward();norms=gradient_norms(model)
                g=torch.nn.utils.clip_grad_norm_(model.parameters(),CFG['gradient_clip'],error_if_nonfinite=True);opt.step()
                values=torch.stack([r.detach(),a.detach(),total.detach(),g.detach()]).cpu().tolist()
                state['step']+=1;state['cursor']=cursor+1
                state['history'].append({'step':state['step'],'epoch':epoch+1,'cursor':cursor+1,'loss_retrieval':values[0],'loss_alignment':values[1],
                   'loss_total':values[2],'gradient_norm':values[3],'lr':opt.param_groups[0]['lr'],**norms})
                if state['step']%CFG['log_every_steps']==0:print(variant,epoch+1,state['step'],values[:3])
                if state['step']%CFG['checkpoint_every_steps']==0:commit()
            influence=alignment_gradient_probe(model,make_batch(PACKAGE['frames']['pilot_train'].iloc[:CFG['batch_size']],CACHE,TOKENIZER))
            metric,pred=evaluate_pilot(model,PACKAGE,CACHE,TOKENIZER)
            require(metric['n_query']==1796,'Full development validation required')
            improved=state['best_mean'] is None or metric['Mean']>state['best_mean']
            if improved:state.update(best_mean=metric['Mean'],best_epoch=epoch+1,bad_epochs=0)
            else:state['bad_epochs']+=1
            state['metrics'].append({'epoch':epoch+1,'metrics':metric,'alignment_gradient_probe':influence,
              'epoch_seconds':time.perf_counter()-start,'peak_vram':torch.cuda.max_memory_allocated() if DEVICE.type=='cuda' else None})
            write_csv(RUN/f'epoch_{epoch+1:02d}_per_query.csv',pred)
            state.update(epoch=epoch+1,cursor=0,finished=epoch+1>=CFG['max_epochs'] or state['bad_epochs']>=CFG['patience'])
            commit(best=improved)
            print(variant,'epoch',epoch+1,'Dev Mean',metric['Mean'],'best',state['best_epoch'])
        return {k:v for k,v in state.items() if k not in ('model','optimizer','rng')}
    except BaseException:
        write_bytes(RUN/'traceback.txt',traceback.format_exc().encode());raise
    finally:
        del model,opt;RUN=ROOT_RUN
        if torch.cuda.is_available():torch.cuda.empty_cache()

@torch.inference_mode()
def export_enhanced_text(model,frame,cache,tokenizer,destination,shard_size=128):
    """Explicit opt-in development-only export, true enhanced sequence [N,77,512] plus mask and query IDs."""
    require(set(frame.official_split)=={'train'},'Official-val export forbidden here')
    require(frame.query_id.is_unique,'Duplicate export query ID');destination=Path(destination)
    export_binding={'model_sha256':tensor_hash(model),'cache':cache.key,'query_ids':frame.query_id.tolist(),'caption_policy':CAPTION_POLICY,'variant':model.variant}
    key=fingerprint(export_binding);root=destination/key;root.mkdir(parents=True,exist_ok=True)
    write_json(root/'index.json',export_binding,immutable=True);model.eval()
    for start in range(0,len(frame),shard_size):
        rows=frame.iloc[start:start+shard_size];chunks=[];masks=[]
        for offset in range(0,len(rows),CFG['eval_batch_size']):
            batch=make_batch(rows.iloc[offset:offset+CFG['eval_batch_size']],cache,tokenizer,training=False)
            h,m=model.enhanced_text_sequence(batch['reference'],batch['modifier']);chunks.append(h.cpu().numpy());masks.append(m.cpu().numpy())
        path=root/f'{start:07d}.npz'
        require(not path.exists(),'Export exists; preserve existing shard and select separate destination')
        temp=path.with_suffix('.partial.npz');np.savez(temp,sequence=np.concatenate(chunks),attention_mask=np.concatenate(masks),query_id=rows.query_id.to_numpy(dtype=str))
        os.replace(temp,path);write_json(path.with_suffix('.json'),{'sha256':file_hash(path),'query_ids':rows.query_id.tolist(),'fingerprint':key})
    return root

def load_best_development(variant):
    """Explicit helper for later development sequence export, never trains a head."""
    root=ROOT_RUN/variant;record=read_json(root/'config.json')
    expected=fingerprint({'variant':variant,'config':CFG,'code':CODE_SHA256,'cache':CACHE.key,'preparation':PACKAGE['binding'],'architecture':ARCHITECTURE})
    require(record['signature']==expected,'Best export run/config/cache differs')
    state=checkpoint_load(root/'best.pt',expected)
    require(state['cache_fingerprint']==CACHE.key,'Best cache fingerprint differs')
    model=initialization_for_variant(BASE,TOKENIZER,variant)
    model.load_state_dict(state['model'],strict=True);model.eval()
    return model


## 2. Kiểm kê → preflight encoder → kế hoạch dung lượng
Cell quota yêu cầu tài khoản Google để đọc dung lượng Drive. Không lưu credential; dừng nếu quota/disk chưa đủ. Cache full-depth 1,84 MB mỗi reference, lớn hơn nhiều final feature. Giữ FP32/12 lớp; chọn runtime disk lớn nếu báo thiếu. Chưa encode toàn bộ ở preflight.

In [ ]:
require(BOOTSTRAP['passed'],'Bootstrap chưa đạt')
init_run();ROOT_RUN=RUN
write_json(RUN/'bootstrap_report.json',BOOTSTRAP)
PACKAGE=load_preparation()
PURE=pure_tests();INFRA=infrastructure_tests()
BASE,TOKENIZER,PROCESSOR,TEMP_MODEL,ARCHITECTURE=load_locked_encoder(PACKAGE)
with stage('encoder_preflight'):
    real_preflight(BASE,TOKENIZER,PROCESSOR,TEMP_MODEL)
del TEMP_MODEL
if torch.cuda.is_available():torch.cuda.empty_cache()
RESOURCE_PLAN=cache_resource_plan(PACKAGE)


## 3. Audit toàn bộ ảnh cần dùng và cache theo lớp
Tái sử dụng dataset đã giải nén. Kaggle chỉ hỏi upload khi auth hiện có thất bại, dùng tên upload thực tế. Ảnh lỗi chặn, không tensor zero. Audit chỉ development cần dùng; không official-val. Cache/mapping kiểm tra một lần trước vòng train.

In [ ]:
with stage('development_cache'):
    cache_model=initialization_for_variant(BASE,TOKENIZER,'full')
    with resource_measure('development_cache_prepare'):
        DRIVE_CACHE=prepare_pilot_cache(PACKAGE,BASE,PROCESSOR,cache_model)
        CACHE=LocalTrainingCache(DRIVE_CACHE)
    del cache_model


## 4. Development đối chứng
Mỗi cấu hình dùng preflight tạm: B1/B4, native adapter, dữ liệu thật, teacher/vision detach, gradient hữu hạn, một AdamW step, save/load rồi khôi phục initialization/RNG. Profiling một batch chỉ là ước lượng compute, không phải thời gian epoch đo thật. Checkpoint chọn bằng toàn bộ development-val, không dùng metric pilot.

In [ ]:
prime_tokens(PACKAGE,TOKENIZER)
RESULTS={}
for variant in CFG['variants']:
    with stage('development_'+variant):
        RESULTS[variant]=development_train(variant)


## 5. API text sequence cho tích hợp sau
`model.enhanced_text_sequence(reference, modifier)` trả hidden sequence sau final LN và attention mask, không composed vector. `export_enhanced_text` ghi shard theo query ID; chỉ gọi có chủ ý với model best đã strict-load đúng signature. Cell này không xuất official-val và không train XAA/DVR.

## 6. Điểm dừng và resume
Runtime mới: chạy từ đầu, cache shard hoàn tất được kiểm tra/tái dùng; runner chọn checkpoint local/Drive có step lớn nhất và cùng signature. Khôi phục optimizer, RNG, cursor; constant LR nên scheduler/scaler=None. Không đổi batch/precision khi resume. Dừng ở development seed42; đánh giá kỹ thuật và so sánh cùng validation trước khi chạy seed43/44. Kết quả chưa chạy để null. GPU/Drive và development đầy đủ chưa được thực thi cho notebook mới; xem báo cáo kiểm thử local đi kèm.

In [ ]:
RUN=ROOT_RUN
summary=[]
for variant in ('conventional','no_alignment','full'):
    p=RUN/variant/'last.json'
    row={'variant':variant,'seed':CFG['seed'],'finished':False,'best_epoch':None,'Dev_Mean':None}
    if p.is_file():
        marker=read_json(p);history=read_json(RUN/variant/'metric_history.json')
        if history:
            best=max(history,key=lambda x:(x['metrics']['Mean'],-x['epoch']))
            row.update(best_epoch=best['epoch'],Dev_Mean=best['metrics']['Mean'],last_epoch=history[-1]['epoch'])
        row['finished']=bool(globals().get('RESULTS',{}).get(variant,{}).get('finished',False))
    summary.append(row)
review={'results':summary,'seeds_not_run':[s for s in (42,43,44) if s!=CFG['seed']],
 'official_val_used':False,'final_training_started':False,'head_training_started':False,
 'evidence_for_three_seed_review':all(r['finished'] for r in summary),
 'next':'Review complete comparisons, gradient/resource/checkpoint reports before explicitly configuring seeds 43/44; no automatic expansion.'}
write_json(RUN/'development_summary.json',review)
print(pd.DataFrame(summary).to_string(index=False))
